# RuNNABLE CODE - version#1 

In [ ]:
import asyncio
from ib_insync import *
import pandas as pd
import numpy as np
import redis
from datetime import datetime
import random

# Connect to Redis
redis_client = redis.Redis(
    host='redis-10429.c280.us-central1-2.gce.redns.redis-cloud.com',
    port=10429,
    username='default',
    password='7zrngrgarW3JXT4cqqoNfdAWc1U26pBB',
    decode_responses=True
)

# Connect to IB
util.startLoop()
ib = IB()
ib.connect('127.0.0.1', 7497, clientId=random.randint(1, 1000))  # Update port and clientId if necessary

tickers = ['AAPL', 'TSLA', 'AMZN', 'GOOG', 'COIN', 'NVDA', 'BA']
contracts = [Stock(ticker, 'SMART', 'USD') for ticker in tickers]

# Ensure contracts are resolved
ib.qualifyContracts(*contracts)

def calculate_rma(df, length, column='close'):
    alpha = 1 / length
    rma = pd.Series(index=df.index, dtype=float)
    rma.iloc[length-1] = df[column].iloc[:length].mean()
    
    for i in range(length, len(df)):
        rma.iloc[i] = (df[column].iloc[i] * alpha) + (rma.iloc[i-1] * (1 - alpha))
    return rma

def calculate_atr(data, period=10):
    data['TR1'] = data['high'] - data['low']
    data['TR2'] = abs(data['high'] - data['close'].shift(1))
    data['TR3'] = abs(data['low'] - data['close'].shift(1))
    data['TR'] = data[['TR1', 'TR2', 'TR3']].max(axis=1)
    data['ATR'] = calculate_rma(data, length=period, column='TR')
    return data

def calculate_supertrend(df, period=10, multiplier=3.0):
    df = calculate_atr(df, period=period)
    high, low, close, atr = df['high'], df['low'], df['close'], df['ATR']
    
    hl2 = (high + low) / 2
    final_upperband = hl2 - (multiplier * atr)
    final_lowerband = hl2 + (multiplier * atr)

    supertrend = pd.DataFrame(index=df.index)
    supertrend['up'] = final_upperband
    supertrend['down'] = final_lowerband
    supertrend['trend'] = 1
    
    for i in range(1, len(df.index)):
        curr, prev = df.index[i], df.index[i-1]
        
        if close[curr] > supertrend.loc[prev, 'up']:
            supertrend.loc[curr, 'up'] = max(supertrend.loc[prev, 'up'], final_upperband[curr])
        else:
            supertrend.loc[curr, 'up'] = final_upperband[curr]
            
        if close[curr] < supertrend.loc[prev, 'down']:
            supertrend.loc[curr, 'down'] = min(supertrend.loc[prev, 'down'], final_lowerband[curr])
        else:
            supertrend.loc[curr, 'down'] = final_lowerband[curr]
            
        supertrend.loc[curr, 'trend'] = 1 if close[curr] > supertrend.loc[prev, 'down'] else \
                                        -1 if close[curr] < supertrend.loc[prev, 'up'] else \
                                        supertrend.loc[prev, 'trend']
            
    supertrend['buy_signal'] = (supertrend['trend'] == 1) & (supertrend['trend'].shift(1) == -1)
    supertrend['sell_signal'] = (supertrend['trend'] == -1) & (supertrend['trend'].shift(1) == 1)
    
    return supertrend

def apply_supertrend(hist, period=10, multiplier=3.0):
    st = calculate_supertrend(hist, period=period, multiplier=multiplier)
    
    hist['SuperTrend_Up'] = np.where(st['trend'] == 1, st['up'], np.nan)
    hist['SuperTrend_Down'] = np.where(st['trend'] == -1, st['down'], np.nan)
    hist['Trend'] = st['trend']
    hist['Buy_Signal'] = st['buy_signal']
    hist['Sell_Signal'] = st['sell_signal']

    hist.drop(columns=['TR1', 'TR2', 'TR3', 'TR'], inplace=True)
    
    return hist

def volume_heatmap(df, length=60, slength=60,
                   threshold_extra_high=4.0,
                   threshold_high=2.5,
                   threshold_medium=1.0,
                   threshold_normal=-0.5):
    """
    Classify volume based on its deviation from the moving average
    """
    # Calculate moving average of volume
    mean = df['volume'].rolling(window=length).mean()

    # Calculate standard deviation of volume
    std = df['volume'].rolling(window=slength).std()

    # Calculate how many standard deviations volume is from mean
    stdbar = (df['volume'] - mean) / std
    df['stdbar'] = stdbar

    # Determine if price closed up or down
    direction = df['close'] > df['open']

    # Classify volume levels
    def classify_volume(row):
        if pd.isna(row['stdbar']):
            return 'Insufficient Data'

        if row['stdbar'] > threshold_extra_high:
            return 'Extra High Volume' + (' Up' if row['direction'] else ' Down')
        elif row['stdbar'] > threshold_high:
            return 'High Volume' + (' Up' if row['direction'] else ' Down')
        elif row['stdbar'] > threshold_medium:
            return 'Medium Volume' + (' Up' if row['direction'] else ' Down')
        elif row['stdbar'] > threshold_normal:
            return 'Normal Volume' + (' Up' if row['direction'] else ' Down')
        else:
            return 'Low Volume' + (' Up' if row['direction'] else ' Down')

    temp_df = pd.DataFrame({
        'stdbar': stdbar,
        'direction': direction
    })

    return temp_df.apply(classify_volume, axis=1)

async def check_and_trade():
    while True:
        # Fetch current positions
        positions = ib.positions()
        position_sizes = {pos.contract.symbol: pos.position for pos in positions}

        # Fetch open orders
        open_orders = ib.reqOpenOrders()
        order_symbols = {order.contract.symbol for order in open_orders}

        for contract in contracts:
            symbol = contract.symbol

            # Fetch historical data
            bars = ib.reqHistoricalData(
                contract,
                endDateTime='',
                durationStr='2 D',
                barSizeSetting='5 mins',
                whatToShow='TRADES',
                useRTH=True,
                formatDate=1
            )

            if not bars:
                print(f"No historical data for {symbol}")
                continue

            # Convert to DataFrame
            df = util.df(bars)
            
            # Ensure required columns are present
            required_columns = ['volume', 'open', 'close']
            if not all(col in df.columns for col in required_columns):
                print(f"Missing columns in data for {symbol}")
                continue

            # Apply volume_heatmap function
            df['Volume_Label'] = volume_heatmap(df)
            df = apply_supertrend(df)

            current_label = df['Volume_Label'].iloc[-1]
            current_row = df.iloc[-1]

            current_position = position_sizes.get(symbol, 0)
            has_pending_order = symbol in order_symbols

            # Define order size
            order_size = 10  # Adjust as needed

            # Initialize trade info from Redis
            trade_key = f"trade:{symbol}"
            trade_data = redis_client.hgetall(trade_key)
            buy_price = float(trade_data.get('price', 0)) if trade_data else 0

            current_price = current_row['close']

            # Check if we have an open position
            if current_position > 0:
                # Calculate profit/loss per share
                profit_loss_per_share = current_price - buy_price

                # Check if profit is $2 or more per share
                if profit_loss_per_share >= 2.0:
                    # Place SELL order to take profit
                    order = MarketOrder('SELL', current_position)
                    trade = ib.placeOrder(contract, order)
                    print(f"Placed TAKE PROFIT SELL order for {symbol} at price {current_price}")

                    # Update trade information in Redis
                    profit_loss_percent = ((current_price - buy_price) / buy_price) * 100
                    trade_info = {
                        'symbol': symbol,
                        'action': 'SELL',
                        'quantity': current_position,
                        'price': current_price,
                        'timestamp': datetime.now().isoformat(),
                        'profit_loss_percent': profit_loss_percent
                    }
                    redis_client.hset(trade_key, mapping=trade_info)
                    print(f"Trade closed for {symbol}. Profit: {profit_loss_per_share:.2f} per share")

                # Check if loss is $1 or more per share
                elif profit_loss_per_share <= -1.0:
                    # Place SELL order to stop loss
                    order = MarketOrder('SELL', current_position)
                    trade = ib.placeOrder(contract, order)
                    print(f"Placed STOP LOSS SELL order for {symbol} at price {current_price}")

                    # Update trade information in Redis
                    profit_loss_percent = ((current_price - buy_price) / buy_price) * 100
                    trade_info = {
                        'symbol': symbol,
                        'action': 'SELL',
                        'quantity': current_position,
                        'price': current_price,
                        'timestamp': datetime.now().isoformat(),
                        'profit_loss_percent': profit_loss_percent
                    }
                    redis_client.hset(trade_key, mapping=trade_info)
                    print(f"Trade closed for {symbol}. Loss: {profit_loss_per_share:.2f} per share")

                else:
                    print(f"Holding position for {symbol}. Current P/L per share: {profit_loss_per_share:.2f}")

            else:
                # Check for buy signal and ensure no pending orders
                if ( current_label == 'Extra High Volume Up' or current_label == 'High Volume Up') and not has_pending_order:
                    # Place BUY order
                    order = MarketOrder('BUY', order_size)
                    trade = ib.placeOrder(contract, order)
                    print(f"Placed BUY order for {symbol} at Volume Label: {current_label}")

                    # Store trade information in Redis
                    trade_info = {
                        'symbol': symbol,
                        'action': 'BUY',
                        'quantity': order_size,
                        'price': current_price,
                        'timestamp': datetime.now().isoformat()
                    }
                    redis_client.hset(trade_key, mapping=trade_info)

                else:
                    if has_pending_order:
                        print(f"Pending order exists for {symbol}. Skipping BUY.")
                    else:
                        print(f"No trade for {symbol}. Current Volume Label: {current_label}")

            # Sleep briefly to avoid pacing violations
            await asyncio.sleep(1)

        # Wait for 5 minutes before next iteration
        print("Waiting for 5 minutes...")
        await asyncio.sleep(300)

# Run the async loop
ib.run(check_and_trade())

# runnable version 2 

In [ ]:
import asyncio
from ib_insync import *
import pandas as pd
import numpy as np
from pymongo import MongoClient, DESCENDING
from datetime import datetime
import random

# Connect to MongoDB
client = MongoClient('mongodb://localhost:27017/')
db = client['trade_database']
trade_collection = db['trades']

# Connect to IB
util.startLoop()
ib = IB()
ib.connect('127.0.0.1', 7497, clientId=random.randint(1, 1000))  # Update port and clientId if necessary

tickers = ['AAPL', 'TSLA', 'AMZN', 'GOOG', 'COIN', 'NVDA', 'BA']
contracts = [Stock(ticker, 'SMART', 'USD') for ticker in tickers]

# Ensure contracts are resolved
ib.qualifyContracts(*contracts)

def calculate_rma(df, length, column='close'):
    alpha = 1 / length
    rma = pd.Series(index=df.index, dtype=float)
    rma.iloc[length-1] = df[column].iloc[:length].mean()
    
    for i in range(length, len(df)):
        rma.iloc[i] = (df[column].iloc[i] * alpha) + (rma.iloc[i-1] * (1 - alpha))
    return rma

def calculate_atr(data, period=10):
    data['TR1'] = data['high'] - data['low']
    data['TR2'] = abs(data['high'] - data['close'].shift(1))
    data['TR3'] = abs(data['low'] - data['close'].shift(1))
    data['TR'] = data[['TR1', 'TR2', 'TR3']].max(axis=1)
    data['ATR'] = calculate_rma(data, length=period, column='TR')
    return data

def calculate_supertrend(df, period=10, multiplier=3.0):
    df = calculate_atr(df, period=period)
    high, low, close, atr = df['high'], df['low'], df['close'], df['ATR']
    
    hl2 = (high + low) / 2
    final_upperband = hl2 - (multiplier * atr)
    final_lowerband = hl2 + (multiplier * atr)

    supertrend = pd.DataFrame(index=df.index)
    supertrend['up'] = final_upperband
    supertrend['down'] = final_lowerband
    supertrend['trend'] = 1
    
    for i in range(1, len(df.index)):
        curr, prev = df.index[i], df.index[i-1]
        
        if close[curr] > supertrend.loc[prev, 'up']:
            supertrend.loc[curr, 'up'] = max(supertrend.loc[prev, 'up'], final_upperband[curr])
        else:
            supertrend.loc[curr, 'up'] = final_upperband[curr]
            
        if close[curr] < supertrend.loc[prev, 'down']:
            supertrend.loc[curr, 'down'] = min(supertrend.loc[prev, 'down'], final_lowerband[curr])
        else:
            supertrend.loc[curr, 'down'] = final_lowerband[curr]
            
        supertrend.loc[curr, 'trend'] = 1 if close[curr] > supertrend.loc[prev, 'down'] else \
                                        -1 if close[curr] < supertrend.loc[prev, 'up'] else \
                                        supertrend.loc[prev, 'trend']
            
    supertrend['buy_signal'] = (supertrend['trend'] == 1) & (supertrend['trend'].shift(1) == -1)
    supertrend['sell_signal'] = (supertrend['trend'] == -1) & (supertrend['trend'].shift(1) == 1)
    
    return supertrend

def apply_supertrend(hist, period=10, multiplier=3.0):
    st = calculate_supertrend(hist, period=period, multiplier=multiplier)
    
    hist['SuperTrend_Up'] = np.where(st['trend'] == 1, st['up'], np.nan)
    hist['SuperTrend_Down'] = np.where(st['trend'] == -1, st['down'], np.nan)
    hist['Trend'] = st['trend']
    hist['Buy_Signal'] = st['buy_signal']
    hist['Sell_Signal'] = st['sell_signal']

    hist.drop(columns=['TR1', 'TR2', 'TR3', 'TR'], inplace=True)
    
    return hist

def volume_heatmap(df, length=60, slength=60,
                   threshold_extra_high=4.0,
                   threshold_high=2.5,
                   threshold_medium=1.0,
                   threshold_normal=-0.5):
    """
    Classify volume based on its deviation from the moving average
    """
    # Calculate moving average of volume
    mean = df['volume'].rolling(window=length).mean()

    # Calculate standard deviation of volume
    std = df['volume'].rolling(window=slength).std()

    # Calculate how many standard deviations volume is from mean
    stdbar = (df['volume'] - mean) / std
    df['stdbar'] = stdbar

    # Determine if price closed up or down
    direction = df['close'] > df['open']

    # Classify volume levels
    def classify_volume(row):
        if pd.isna(row['stdbar']):
            return 'Insufficient Data'

        if row['stdbar'] > threshold_extra_high:
            return 'Extra High Volume' + (' Up' if row['direction'] else ' Down')
        elif row['stdbar'] > threshold_high:
            return 'High Volume' + (' Up' if row['direction'] else ' Down')
        elif row['stdbar'] > threshold_medium:
            return 'Medium Volume' + (' Up' if row['direction'] else ' Down')
        elif row['stdbar'] > threshold_normal:
            return 'Normal Volume' + (' Up' if row['direction'] else ' Down')
        else:
            return 'Low Volume' + (' Up' if row['direction'] else ' Down')

    temp_df = pd.DataFrame({
        'stdbar': stdbar,
        'direction': direction
    })

    return temp_df.apply(classify_volume, axis=1)

async def check_and_trade():
    while True:
        # Fetch current positions
        positions = ib.positions()
        position_sizes = {pos.contract.symbol: pos.position for pos in positions}

        # Fetch open orders
        open_orders = ib.reqOpenOrders()
        order_symbols = {order.contract.symbol for order in open_orders}

        for contract in contracts:
            symbol = contract.symbol

            # Fetch historical data
            bars = ib.reqHistoricalData(
                contract,
                endDateTime='',
                durationStr='2 D',
                barSizeSetting='5 mins',
                whatToShow='TRADES',
                useRTH=True,
                formatDate=1
            )

            if not bars:
                print(f"No historical data for {symbol}")
                continue

            # Convert to DataFrame
            df = util.df(bars)
            
            # Ensure required columns are present
            required_columns = ['volume', 'open', 'close']
            if not all(col in df.columns for col in required_columns):
                print(f"Missing columns in data for {symbol}")
                continue

            # Apply volume_heatmap function
            df['Volume_Label'] = volume_heatmap(df)
            df = apply_supertrend(df)

            current_label = df['Volume_Label'].iloc[-1]
            current_row = df.iloc[-1]

            current_position = position_sizes.get(symbol, 0)
            has_pending_order = symbol in order_symbols

            # Define order size
            order_size = 10  # Adjust as needed

            # Fetch the last BUY trade to get the buy price
            trade_data = trade_collection.find_one(
                {'symbol': symbol, 'action': 'BUY'},
                sort=[('timestamp', DESCENDING)]
            )
            buy_price = float(trade_data.get('price', 0)) if trade_data else 0

            current_price = current_row['close']

            # Check if we have an open position
            if current_position > 0:
                # Calculate profit/loss per share
                profit_loss_per_share = current_price - buy_price

                # Check if profit is $2 or more per share
                if profit_loss_per_share >= 2.0:
                    # Place SELL order to take profit
                    order = MarketOrder('SELL', current_position)
                    trade = ib.placeOrder(contract, order)
                    print(f"Placed TAKE PROFIT SELL order for {symbol} at price {current_price}")

                    # Calculate profit/loss percent
                    profit_loss_percent = ((current_price - buy_price) / buy_price) * 100

                    # Update trade information in MongoDB
                    trade_info = {
                        'symbol': symbol,
                        'action': 'SELL',
                        'quantity': current_position,
                        'price': current_price,
                        'timestamp': datetime.now(),
                        'profit_loss_percent': profit_loss_percent
                    }
                    trade_collection.insert_one(trade_info)
                    print(f"Trade closed for {symbol}. Profit: {profit_loss_per_share:.2f} per share")

                # Check if loss is $1 or more per share
                elif profit_loss_per_share <= -1.0:
                    # Place SELL order to stop loss
                    order = MarketOrder('SELL', current_position)
                    trade = ib.placeOrder(contract, order)
                    print(f"Placed STOP LOSS SELL order for {symbol} at price {current_price}")

                    # Calculate profit/loss percent
                    profit_loss_percent = ((current_price - buy_price) / buy_price) * 100

                    # Update trade information in MongoDB
                    trade_info = {
                        'symbol': symbol,
                        'action': 'SELL',
                        'quantity': current_position,
                        'price': current_price,
                        'timestamp': datetime.now(),
                        'profit_loss_percent': profit_loss_percent
                    }
                    trade_collection.insert_one(trade_info)
                    print(f"Trade closed for {symbol}. Loss: {profit_loss_per_share:.2f} per share")

                else:
                    print(f"Holding position for {symbol}. Current P/L per share: {profit_loss_per_share:.2f}")

            else:
                # Check for buy signal and ensure no pending orders
                if (current_label == 'Extra High Volume Up' or current_label == 'High Volume Up') and not has_pending_order:
                    # Get current bid price
                    ticker =  ib.reqMktData(contract)    #ib.reqMktData(contract, '', False, False)
                    await asyncio.sleep(1)  # Wait for market data to arrive
                    bid_price = ticker.bid

                    if bid_price is not None and bid_price > 0:
                        # Place a LIMIT BUY order at bid price
                        order = LimitOrder('BUY', order_size, bid_price)
                        trade = ib.placeOrder(contract, order)
                        print(f"Placed BUY LIMIT order for {symbol} at bid price {bid_price}")

                        # Store trade information in MongoDB
                        trade_info = {
                            'symbol': symbol,
                            'action': 'BUY',
                            'quantity': order_size,
                            'price': bid_price,
                            'timestamp': datetime.now()
                        }
                        trade_collection.insert_one(trade_info)
                    else:
                        print(f"Could not retrieve bid price for {symbol}. Skipping BUY.")
                else:
                    if has_pending_order:
                        print(f"Pending order exists for {symbol}. Skipping BUY.")
                    else:
                        print(f"No trade for {symbol}. Current Volume Label: {current_label}")

            # Cancel market data subscription to prevent memory leaks
            ib.cancelMktData(contract)

            # Sleep briefly to avoid pacing violations
            await asyncio.sleep(1)

        # Wait for 5 minutes before next iteration
        print("Waiting for 5 minutes...")
        await asyncio.sleep(300)

# Run the async loop
ib.run(check_and_trade())

# BACKTESTING CODE

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta

def calculate_rma(series, length):
    """
    Calculate the Relative Moving Average (RMA) of a given series.
    """
    rma = series.ewm(alpha=1/length, adjust=False).mean()
    return rma

def calculate_atr(df, period=10):
    """
    Calculate the Average True Range (ATR) using the RMA of the True Range.
    """
    df['H-L'] = df['High'] - df['Low']
    df['H-PC'] = abs(df['High'] - df['Close'].shift(1))
    df['L-PC'] = abs(df['Low'] - df['Close'].shift(1))
    df['TR'] = df[['H-L', 'H-PC', 'L-PC']].max(axis=1)
    df['ATR'] = calculate_rma(df['TR'], length=period)
    df.drop(['H-L', 'H-PC', 'L-PC'], axis=1, inplace=True)
    return df

def calculate_supertrend(df, period=10, multiplier=3):
    """
    Calculate SuperTrend indicator.
    """
    df = calculate_atr(df, period)
    hl2 = (df['High'] + df['Low']) / 2
    df['Basic Upper Band'] = hl2 + (multiplier * df['ATR'])
    df['Basic Lower Band'] = hl2 - (multiplier * df['ATR'])
    df['Final Upper Band'] = df['Basic Upper Band']
    df['Final Lower Band'] = df['Basic Lower Band']
    df['SuperTrend'] = np.nan
    df['Trend'] = 0

    for i in range(1, len(df)):
        curr = df.index[i]
        prev = df.index[i - 1]

        # Final Upper Band
        if (df['Basic Upper Band'].iloc[i] < df['Final Upper Band'].iloc[prev]) or (df['Close'].iloc[prev] > df['Final Upper Band'].iloc[prev]):
            df['Final Upper Band'].iloc[i] = df['Basic Upper Band'].iloc[i]
        else:
            df['Final Upper Band'].iloc[i] = df['Final Upper Band'].iloc[prev]

        # Final Lower Band
        if (df['Basic Lower Band'].iloc[i] > df['Final Lower Band'].iloc[prev]) or (df['Close'].iloc[prev] < df['Final Lower Band'].iloc[prev]):
            df['Final Lower Band'].iloc[i] = df['Basic Lower Band'].iloc[i]
        else:
            df['Final Lower Band'].iloc[i] = df['Final Lower Band'].iloc[prev]

        # SuperTrend and Trend
        if df['Close'].iloc[i - 1] <= df['Final Upper Band'].iloc[i - 1]:
            if df['Close'].iloc[i] > df['Final Upper Band'].iloc[i]:
                df['Trend'].iloc[i] = 1
            else:
                df['Trend'].iloc[i] = -1
        else:
            if df['Close'].iloc[i] < df['Final Lower Band'].iloc[i]:
                df['Trend'].iloc[i] = -1
            else:
                df['Trend'].iloc[i] = 1

        # SuperTrend value
        if df['Trend'].iloc[i] == 1:
            df['SuperTrend'].iloc[i] = df['Final Lower Band'].iloc[i]
        else:
            df['SuperTrend'].iloc[i] = df['Final Upper Band'].iloc[i]

    df['SuperTrend'] = df['SuperTrend'].fillna(0)
    df['Buy Signal'] = (df['Trend'] == 1) & (df['Trend'].shift(1) == -1)
    df['Sell Signal'] = (df['Trend'] == -1) & (df['Trend'].shift(1) == 1)
    df.drop(['Basic Upper Band', 'Basic Lower Band'], axis=1, inplace=True)
    return df

def volume_heatmap(df, length=60, slength=60,
                   threshold_extra_high=4.0,
                   threshold_high=2.5,
                   threshold_medium=1.0,
                   threshold_normal=-0.5):
    """
    Classify volume based on its deviation from the moving average.
    """
    # Ensure there are enough data points
    df['Volume_MA'] = df['Volume'].rolling(window=length).mean()
    df['Volume_STD'] = df['Volume'].rolling(window=slength).std()
    df['StdBar'] = (df['Volume'] - df['Volume_MA']) / df['Volume_STD']
    df['Direction'] = df['Close'] > df['Open']

    conditions = [
        df['StdBar'] > threshold_extra_high,
        df['StdBar'].between(threshold_high, threshold_extra_high),
        df['StdBar'].between(threshold_medium, threshold_high),
        df['StdBar'].between(threshold_normal, threshold_medium),
        df['StdBar'] <= threshold_normal
    ]
    choices = ['Extra High Volume', 'High Volume', 'Medium Volume', 'Normal Volume', 'Low Volume']
    df['Volume_Label'] = np.select(conditions, choices, default='Insufficient Data')
    df['Volume_Label'] += np.where(df['Direction'], ' Up', ' Down')

    df.drop(['Volume_MA', 'Volume_STD', 'StdBar', 'Direction'], axis=1, inplace=True)
    return df
def replace_dataframe_header(df):
  new_columns = [ "Close", "High", "Low", "Open", "Volume"]

  # Check if the dataframe has at least the correct number of columns.
  if len(df.columns) < len(new_columns):
    raise ValueError(f"DataFrame must have at least {len(new_columns)} columns.")

  df.columns = new_columns[:len(df.columns)] # handle cases where the dataframe has more columns than the new header.

  return df

print(f"Backtesting for {ticker}")
    # Fetch historical data
data1 = yf.download(ticker, period='1mo', interval='5m')
data = replace_dataframe_header(data1.copy())  

data.reset_index(inplace=True)
    # Convert column names to match the code
    #data.columns = [col.capitalize() for col in data.columns]

    # Ensure required columns are present
required_columns = ['Datetime', 'Open', 'High', 'Low', 'Close', 'Volume']
if not all(col in data.columns for col in required_columns):
    print(f"Missing columns in data for {ticker}")

print(data.head)
    
#data = calculate_supertrend(data)
data = volume_heatmap(data)

# Runnable code 3 - QQE / SSL and SuperTrend and heatmap

In [ ]:
import asyncio
from ib_insync import *
import pandas as pd
import numpy as np
import redis
from datetime import datetime
import random

# Connect to Redis
redis_client = redis.Redis(
    host='redis-10429.c280.us-central1-2.gce.redns.redis-cloud.com',
    port=10429,
    username='default',
    password='7zrngrgarW3JXT4cqqoNfdAWc1U26pBB',
    decode_responses=True
)

# Connect to IB
util.startLoop()
ib = IB()
ib.connect('127.0.0.1', 7497, clientId=random.randint(1, 1000))  # Update port and clientId if necessary

tickers = ['AAPL', 'TSLA', 'AMZN', 'GOOG', 'COIN', 'NVDA', 'BA']
contracts = [Stock(ticker, 'SMART', 'USD') for ticker in tickers]

# Ensure contracts are resolved
ib.qualifyContracts(*contracts)

def calculate_qqe_signal(df,
                         rsi_length_primary=6,
                         rsi_smoothing_primary=5,
                         qqe_factor_primary=3.0,
                         threshold_primary=3.0,
                         rsi_length_secondary=6,
                         rsi_smoothing_secondary=5,
                         qqe_factor_secondary=1.61,
                         threshold_secondary=3.0,
                         bollinger_length=50,
                         bollinger_multiplier=0.35):
    """
    Calculates the QQE (Quantitative Qualitative Estimation) signal and adds it as a column to the DataFrame.

    Parameters:
        df (pd.DataFrame): Input DataFrame with a 'close' column.
        rsi_length_primary (int): RSI length for the primary QQE.
        rsi_smoothing_primary (int): Smoothing factor for the primary RSI.
        qqe_factor_primary (float): QQE factor for the primary QQE.
        threshold_primary (float): Threshold for the primary QQE.
        rsi_length_secondary (int): RSI length for the secondary QQE.
        rsi_smoothing_secondary (int): Smoothing factor for the secondary RSI.
        qqe_factor_secondary (float): QQE factor for the secondary QQE.
        threshold_secondary (float): Threshold for the secondary QQE.
        bollinger_length (int): Length for the Bollinger Bands.
        bollinger_multiplier (float): Multiplier for the Bollinger Bands.

    Returns:
        pd.DataFrame: DataFrame with added columns 'Primary RSI', 'Secondary RSI', 'Bollinger Upper', 'Bollinger Lower', and 'QQE_Signal'.
    """

    def calculate_qqe(rsi_length, smoothing_factor, qqe_factor, source):
        wilders_length = rsi_length * 2 - 1

        delta = source.diff()
        up = delta.clip(lower=0)
        down = -delta.clip(upper=0)
        up_avg = up.ewm(alpha=1 / rsi_length, adjust=False).mean()
        down_avg = down.ewm(alpha=1 / rsi_length, adjust=False).mean()
        rs = up_avg / down_avg
        rsi = 100 - (100 / (1 + rs))

        smoothed_rsi = rsi.ewm(span=smoothing_factor, adjust=False).mean()

        atr_rsi = (smoothed_rsi - smoothed_rsi.shift(1)).abs()
        smoothed_atr_rsi = atr_rsi.ewm(span=wilders_length, adjust=False).mean()
        dynamic_atr_rsi = smoothed_atr_rsi * qqe_factor

        long_band = pd.Series(index=source.index)
        short_band = pd.Series(index=source.index)
        trend_direction = pd.Series(index=source.index, dtype='int')

        long_band.iloc[0] = smoothed_rsi.iloc[0]
        short_band.iloc[0] = smoothed_rsi.iloc[0]
        trend_direction.iloc[0] = 0

        for i in range(1, len(source)):
            atr_delta = dynamic_atr_rsi.iloc[i]
            new_long_band = smoothed_rsi.iloc[i] - atr_delta
            new_short_band = smoothed_rsi.iloc[i] + atr_delta

            prev_long_band = long_band.iloc[i - 1]
            prev_short_band = short_band.iloc[i - 1]
            prev_smoothed_rsi = smoothed_rsi.iloc[i - 1]

            if prev_smoothed_rsi > prev_long_band and smoothed_rsi.iloc[i] > prev_long_band:
                long_band.iloc[i] = max(prev_long_band, new_long_band)
            else:
                long_band.iloc[i] = new_long_band

            if prev_smoothed_rsi < prev_short_band and smoothed_rsi.iloc[i] < prev_short_band:
                short_band.iloc[i] = min(prev_short_band, new_short_band)
            else:
                short_band.iloc[i] = new_short_band

            if smoothed_rsi.iloc[i] > prev_short_band and prev_smoothed_rsi <= prev_short_band:
                trend_direction.iloc[i] = 1
            elif smoothed_rsi.iloc[i] < prev_long_band and prev_smoothed_rsi >= prev_long_band:
                trend_direction.iloc[i] = -1
            else:
                trend_direction.iloc[i] = trend_direction.iloc[i - 1]

        qqe_trend_line = pd.Series(index=source.index)
        for i in range(len(source)):
            if trend_direction.iloc[i] == 1:
                qqe_trend_line.iloc[i] = long_band.iloc[i]
            else:
                qqe_trend_line.iloc[i] = short_band.iloc[i]

        return qqe_trend_line, smoothed_rsi

    # Primary QQE Calculations
    primary_qqe_trend_line, primary_rsi = calculate_qqe(rsi_length_primary, rsi_smoothing_primary,
                                                          qqe_factor_primary, df['close'])

    # Secondary QQE Calculations
    secondary_qqe_trend_line, secondary_rsi = calculate_qqe(rsi_length_secondary, rsi_smoothing_secondary,
                                                              qqe_factor_secondary, df['close'])

    # Adjustments
    primary_qqe_trend_line_adj = primary_qqe_trend_line - 50
    bollinger_basis = primary_qqe_trend_line_adj.rolling(window=bollinger_length).mean()
    bollinger_deviation = bollinger_multiplier * primary_qqe_trend_line_adj.rolling(window=bollinger_length).std()
    bollinger_upper = bollinger_basis + bollinger_deviation
    bollinger_lower = bollinger_basis - bollinger_deviation

    primary_rsi_adj = primary_rsi - 50
    secondary_rsi_adj = secondary_rsi - 50

    # Add calculated values to the DataFrame
    df['Primary RSI'] = primary_rsi_adj
    df['Secondary RSI'] = secondary_rsi_adj
    df['Bollinger Upper'] = bollinger_upper
    df['Bollinger Lower'] = bollinger_lower

    # Generate trading signals
    df['QQE_Signal'] = np.where((secondary_rsi_adj > threshold_secondary) & (primary_rsi_adj > bollinger_upper),
                                'Up',
                                np.where((secondary_rsi_adj < -threshold_secondary) & (primary_rsi_adj < bollinger_lower),
                                         'Down', None))

    return df

def add_ssl_channel_color(df, atr_period=14, atr_mult=1, atr_smoothing='WMA', baseline_length=60):
    """
    Adds an "SSL_Channel_Color" column to the given DataFrame based on SSL Channel logic.

    Parameters:
        df (pd.DataFrame): Input DataFrame with 'high', 'low', and 'close' columns.
        atr_period (int): Period for ATR calculation.
        atr_mult (float): Multiplier for ATR bands.
        atr_smoothing (str): Smoothing type for ATR ('WMA' or 'SMA').
        baseline_length (int): Length for the baseline HMA calculation.

    Returns:
        pd.DataFrame: Updated DataFrame with "SSL_Channel_Color" column.
    """
    def wma(series, length):
        """Weighted Moving Average (WMA)"""
        weights = np.arange(1, length + 1)
        return series.rolling(length).apply(lambda prices: np.dot(prices, weights) / weights.sum(), raw=True)

    def hma(series, length):
        """Hull Moving Average (HMA)"""
        half_length = int(length / 2)
        sqrt_length = int(np.sqrt(length))
        wma1 = wma(series, half_length)
        wma2 = wma(series, length)
        diff = 2 * wma1 - wma2
        return wma(diff, sqrt_length)

    def calculate_atr(data, length=14, smoothing='WMA'):
        """Average True Range (ATR) calculation"""
        high = data['high']
        low = data['low']
        close = data['close']
        
        tr1 = high - low
        tr2 = abs(high - close.shift(1))
        tr3 = abs(low - close.shift(1))
        tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)

        if smoothing == 'WMA':
            atr = wma(tr, length)
        else:
            # Default to simple moving average if not WMA
            atr = tr.rolling(window=length).mean()
        return atr

    # Ensure no NaN values and reset index
    df.dropna(inplace=True)
    df.reset_index(drop=True, inplace=True)

    # Baseline calculation using HMA
    baseline_hma = hma(df['close'], baseline_length)

    # ATR calculation
    atr_values = calculate_atr(df, length=atr_period, smoothing=atr_smoothing)
    upper_band = df['close'] + atr_values * atr_mult
    lower_band = df['close'] - atr_values * atr_mult

    # SSL Channel calculations
    ema_high = hma(df['high'], baseline_length)
    ema_low = hma(df['low'], baseline_length)

    hlv = np.where(df['close'] > ema_high, 1, np.where(df['close'] < ema_low, -1, np.nan))
    hlv = pd.Series(hlv).ffill()
    
    ssl_down = np.where(hlv < 0, ema_high, ema_low)
    ssl_down_series = pd.Series(ssl_down, index=df.index)

    # Determine SSL Channel Color
    ssl_channel_color = np.where(df['close'] > ssl_down_series, 'blue', 
                                 np.where(df['close'] < ssl_down_series, 'red', np.nan))
    
    # Add SSL_Channel_Color to DataFrame
    df['SSL_Channel_Color'] = ssl_channel_color

    return df

def calculate_rma(df, length, column='close'):
    alpha = 1 / length
    rma = pd.Series(index=df.index, dtype=float)
    rma.iloc[length-1] = df[column].iloc[:length].mean()
    
    for i in range(length, len(df)):
        rma.iloc[i] = (df[column].iloc[i] * alpha) + (rma.iloc[i-1] * (1 - alpha))
    return rma

def calculate_atr(data, period=10):
    data['TR1'] = data['high'] - data['low']
    data['TR2'] = abs(data['high'] - data['close'].shift(1))
    data['TR3'] = abs(data['low'] - data['close'].shift(1))
    data['TR'] = data[['TR1', 'TR2', 'TR3']].max(axis=1)
    data['ATR'] = calculate_rma(data, length=period, column='TR')
    return data

def calculate_supertrend(df, period=10, multiplier=3.0):
    df = calculate_atr(df, period=period)
    high, low, close, atr = df['high'], df['low'], df['close'], df['ATR']
    
    hl2 = (high + low) / 2
    final_upperband = hl2 - (multiplier * atr)
    final_lowerband = hl2 + (multiplier * atr)

    supertrend = pd.DataFrame(index=df.index)
    supertrend['up'] = final_upperband
    supertrend['down'] = final_lowerband
    supertrend['trend'] = 1
    
    for i in range(1, len(df.index)):
        curr, prev = df.index[i], df.index[i-1]
        
        if close[curr] > supertrend.loc[prev, 'up']:
            supertrend.loc[curr, 'up'] = max(supertrend.loc[prev, 'up'], final_upperband[curr])
        else:
            supertrend.loc[curr, 'up'] = final_upperband[curr]
            
        if close[curr] < supertrend.loc[prev, 'down']:
            supertrend.loc[curr, 'down'] = min(supertrend.loc[prev, 'down'], final_lowerband[curr])
        else:
            supertrend.loc[curr, 'down'] = final_lowerband[curr]
            
        supertrend.loc[curr, 'trend'] = 1 if close[curr] > supertrend.loc[prev, 'down'] else \
                                        -1 if close[curr] < supertrend.loc[prev, 'up'] else \
                                        supertrend.loc[prev, 'trend']
            
    supertrend['buy_signal'] = (supertrend['trend'] == 1) & (supertrend['trend'].shift(1) == -1)
    supertrend['sell_signal'] = (supertrend['trend'] == -1) & (supertrend['trend'].shift(1) == 1)
    
    return supertrend

def apply_supertrend(hist, period=10, multiplier=3.0):
    st = calculate_supertrend(hist, period=period, multiplier=multiplier)
    
    hist['SuperTrend_Up'] = np.where(st['trend'] == 1, st['up'], np.nan)
    hist['SuperTrend_Down'] = np.where(st['trend'] == -1, st['down'], np.nan)
    hist['Trend'] = st['trend']
    hist['Buy_Signal'] = st['buy_signal']
    hist['Sell_Signal'] = st['sell_signal']

    hist.drop(columns=['TR1', 'TR2', 'TR3', 'TR'], inplace=True)
    
    return hist

def volume_heatmap(df, length=60, slength=60,
                   threshold_extra_high=4.0,
                   threshold_high=2.5,
                   threshold_medium=1.0,
                   threshold_normal=-0.5):
    """
    Classify volume based on its deviation from the moving average
    """
    # Calculate moving average of volume
    mean = df['volume'].rolling(window=length).mean()

    # Calculate standard deviation of volume
    std = df['volume'].rolling(window=slength).std()

    # Calculate how many standard deviations volume is from mean
    stdbar = (df['volume'] - mean) / std
    df['stdbar'] = stdbar

    # Determine if price closed up or down
    direction = df['close'] > df['open']

    # Classify volume levels
    def classify_volume(row):
        if pd.isna(row['stdbar']):
            return 'Insufficient Data'

        if row['stdbar'] > threshold_extra_high:
            return 'Extra High Volume' + (' Up' if row['direction'] else ' Down')
        elif row['stdbar'] > threshold_high:
            return 'High Volume' + (' Up' if row['direction'] else ' Down')
        elif row['stdbar'] > threshold_medium:
            return 'Medium Volume' + (' Up' if row['direction'] else ' Down')
        elif row['stdbar'] > threshold_normal:
            return 'Normal Volume' + (' Up' if row['direction'] else ' Down')
        else:
            return 'Low Volume' + (' Up' if row['direction'] else ' Down')

    temp_df = pd.DataFrame({
        'stdbar': stdbar,
        'direction': direction
    })

    return temp_df.apply(classify_volume, axis=1)

async def check_and_trade():
    while True:
        # Fetch current positions
        positions = ib.positions()
        position_sizes = {pos.contract.symbol: pos.position for pos in positions}

        # Fetch open orders
        open_orders = ib.reqOpenOrders()
        order_symbols = {order.contract.symbol for order in open_orders}

        for contract in contracts:
            symbol = contract.symbol

            # Fetch historical data
            bars = ib.reqHistoricalData(
                contract,
                endDateTime='',
                durationStr='2 D',
                barSizeSetting='5 mins',
                whatToShow='TRADES',
                useRTH=True,
                formatDate=1
            )

            if not bars:
                print(f"No historical data for {symbol}")
                continue

            # Convert to DataFrame
            df = util.df(bars)
            
            # Ensure required columns are present
            required_columns = ['volume', 'open', 'close']
            if not all(col in df.columns for col in required_columns):
                print(f"Missing columns in data for {symbol}")
                continue

            # Apply volume_heatmap function
            df = add_ssl_channel_color(df)
            df['Volume_Label'] = volume_heatmap(df)
            df = calculate_qqe_signal(df)
            df = apply_supertrend(df)


            current_label = df['Volume_Label'].iloc[-1]
            current_row = df.iloc[-1]
            #print(current_row)

            current_position = position_sizes.get(symbol, 0)
            has_pending_order = symbol in order_symbols

            # Define order size
            order_size = 10  # Adjust as needed

            # Initialize trade info from Redis
            trade_key = f"trade:{symbol}"
            trade_data = redis_client.hgetall(trade_key)
            buy_price = float(trade_data.get('price', 0)) if trade_data else 0

            current_price = current_row['close']

            # Check if we have an open position
            if current_position > 0:
                # Calculate profit/loss per share
                profit_loss_per_share = current_price - buy_price

                # Check if profit is $2 or more per share
                if profit_loss_per_share >= 1.0:
                    # Place SELL order to take profit
                    order = MarketOrder('SELL', current_position)
                    trade = ib.placeOrder(contract, order)
                    print(f"Placed TAKE PROFIT SELL order for {symbol} at price {current_price}")

                    # Update trade information in Redis
                    profit_loss_percent = ((current_price - buy_price) / buy_price) * 100
                    trade_info = {
                        'symbol': symbol,
                        'action': 'SELL',
                        'quantity': current_position,
                        'price': current_price,
                        'timestamp': datetime.now().isoformat(),
                        'profit_loss_percent': profit_loss_percent
                    }
                    redis_client.hset(trade_key, mapping=trade_info)
                    print(f"Trade closed for {symbol}. Profit: {profit_loss_per_share:.2f} per share")

                # Check if loss is $1 or more per share
                elif profit_loss_per_share <= -1.0:
                    # Place SELL order to stop loss
                    order = MarketOrder('SELL', current_position)
                    trade = ib.placeOrder(contract, order)
                    print(f"Placed STOP LOSS SELL order for {symbol} at price {current_price}")

                    # Update trade information in Redis
                    profit_loss_percent = ((current_price - buy_price) / buy_price) * 100
                    trade_info = {
                        'symbol': symbol,
                        'action': 'SELL',
                        'quantity': current_position,
                        'price': current_price,
                        'timestamp': datetime.now().isoformat(),
                        'profit_loss_percent': profit_loss_percent
                    }
                    redis_client.hset(trade_key, mapping=trade_info)
                    print(f"Trade closed for {symbol}. Loss: {profit_loss_per_share:.2f} per share")

                else:
                    print(f"Holding position for {symbol}. Current P/L per share: {profit_loss_per_share:.2f}")

            else:
                # Check for buy signal and ensure no pending orders
                #SSL_Channel_Color == 'blue' and Trend == -1 and QQE_Signal == 'Up'
                if ( current_row['SSL_Channel_Color'] == 'blue' and current_row['Trend'] == -1 and current_row['QQE_Signal'] == 'Up') and not has_pending_order:
                #if ( current_label == 'Extra High Volume Up' or current_label == 'High Volume Up') and not has_pending_order:
                    # Place BUY order
                    order = MarketOrder('BUY', order_size)
                    trade = ib.placeOrder(contract, order)
                    print(f"Placed BUY order for {symbol} at Volume Label: {current_label}")
                    print(current_row)

                    # Store trade information in Redis
                    trade_info = {
                        'symbol': symbol,
                        'action': 'BUY',
                        'quantity': order_size,
                        'price': current_price,
                        'timestamp': datetime.now().isoformat()
                    }
                    redis_client.hset(trade_key, mapping=trade_info)

                else:
                    if has_pending_order:
                        print(f"Pending order exists for {symbol}. Skipping BUY.")
                    else:
                        print(f"No trade for {symbol}. Current Volume Label: {current_label}")

            # Sleep briefly to avoid pacing violations
            await asyncio.sleep(1)

        # Wait for 5 minutes before next iteration
        print("Waiting for 5 minutes...")
        await asyncio.sleep(300)

# Run the async loop
ib.run(check_and_trade())

# CODE 4 - updated with mongo load 

In [ ]:
import asyncio
from ib_insync import *
import pandas as pd
import numpy as np
from datetime import datetime
import random
from pymongo import MongoClient
from bson import ObjectId


# Connect to MongoDB
client = MongoClient('mongodb://localhost:27017/')
db = client['trading_db']
trades_collection = db['trades']

# Connect to IB
util.startLoop()
ib = IB()
ib.connect('127.0.0.1', 7497, clientId=random.randint(1, 1000))  # Update port and clientId if necessary

tickers = ['AAPL', 'TSLA', 'AMZN', 'GOOG', 'COIN', 'NVDA', 'BA']
contracts = [Stock(ticker, 'SMART', 'USD') for ticker in tickers]

# Ensure contracts are resolved
ib.qualifyContracts(*contracts)

# Helper function to create trade document
def create_trade_document(symbol, direction, entry_price, quantity):
    return {
        'instrument': symbol,
        'direction': direction,
        'entry_price': entry_price,
        'quantity': quantity,
        'entry_timestamp': datetime.now(),
        'order_id': str(ObjectId()),
        'exit_price': None,
        'exit_timestamp': None,
        'reason': None,
        'current_pnl': 0,
        'realized_pnl': 0,
        'status': 'open',
        'created_at': datetime.now(),
        'updated_at': datetime.now()
    }

# Helper function to update trade document
def update_trade_document(trade_id, updates):
    updates['updated_at'] = datetime.now()
    trades_collection.update_one(
        {'_id': ObjectId(trade_id)},
        {'$set': updates}
    )

def calculate_qqe_signal(df,
                         rsi_length_primary=6,
                         rsi_smoothing_primary=5,
                         qqe_factor_primary=3.0,
                         threshold_primary=3.0,
                         rsi_length_secondary=6,
                         rsi_smoothing_secondary=5,
                         qqe_factor_secondary=1.61,
                         threshold_secondary=3.0,
                         bollinger_length=50,
                         bollinger_multiplier=0.35):
    """
    Calculates the QQE (Quantitative Qualitative Estimation) signal and adds it as a column to the DataFrame.

    Parameters:
        df (pd.DataFrame): Input DataFrame with a 'close' column.
        rsi_length_primary (int): RSI length for the primary QQE.
        rsi_smoothing_primary (int): Smoothing factor for the primary RSI.
        qqe_factor_primary (float): QQE factor for the primary QQE.
        threshold_primary (float): Threshold for the primary QQE.
        rsi_length_secondary (int): RSI length for the secondary QQE.
        rsi_smoothing_secondary (int): Smoothing factor for the secondary RSI.
        qqe_factor_secondary (float): QQE factor for the secondary QQE.
        threshold_secondary (float): Threshold for the secondary QQE.
        bollinger_length (int): Length for the Bollinger Bands.
        bollinger_multiplier (float): Multiplier for the Bollinger Bands.

    Returns:
        pd.DataFrame: DataFrame with added columns 'Primary RSI', 'Secondary RSI', 'Bollinger Upper', 'Bollinger Lower', and 'QQE_Signal'.
    """

    def calculate_qqe(rsi_length, smoothing_factor, qqe_factor, source):
        wilders_length = rsi_length * 2 - 1

        delta = source.diff()
        up = delta.clip(lower=0)
        down = -delta.clip(upper=0)
        up_avg = up.ewm(alpha=1 / rsi_length, adjust=False).mean()
        down_avg = down.ewm(alpha=1 / rsi_length, adjust=False).mean()
        rs = up_avg / down_avg
        rsi = 100 - (100 / (1 + rs))

        smoothed_rsi = rsi.ewm(span=smoothing_factor, adjust=False).mean()

        atr_rsi = (smoothed_rsi - smoothed_rsi.shift(1)).abs()
        smoothed_atr_rsi = atr_rsi.ewm(span=wilders_length, adjust=False).mean()
        dynamic_atr_rsi = smoothed_atr_rsi * qqe_factor

        long_band = pd.Series(index=source.index)
        short_band = pd.Series(index=source.index)
        trend_direction = pd.Series(index=source.index, dtype='int')

        long_band.iloc[0] = smoothed_rsi.iloc[0]
        short_band.iloc[0] = smoothed_rsi.iloc[0]
        trend_direction.iloc[0] = 0

        for i in range(1, len(source)):
            atr_delta = dynamic_atr_rsi.iloc[i]
            new_long_band = smoothed_rsi.iloc[i] - atr_delta
            new_short_band = smoothed_rsi.iloc[i] + atr_delta

            prev_long_band = long_band.iloc[i - 1]
            prev_short_band = short_band.iloc[i - 1]
            prev_smoothed_rsi = smoothed_rsi.iloc[i - 1]

            if prev_smoothed_rsi > prev_long_band and smoothed_rsi.iloc[i] > prev_long_band:
                long_band.iloc[i] = max(prev_long_band, new_long_band)
            else:
                long_band.iloc[i] = new_long_band

            if prev_smoothed_rsi < prev_short_band and smoothed_rsi.iloc[i] < prev_short_band:
                short_band.iloc[i] = min(prev_short_band, new_short_band)
            else:
                short_band.iloc[i] = new_short_band

            if smoothed_rsi.iloc[i] > prev_short_band and prev_smoothed_rsi <= prev_short_band:
                trend_direction.iloc[i] = 1
            elif smoothed_rsi.iloc[i] < prev_long_band and prev_smoothed_rsi >= prev_long_band:
                trend_direction.iloc[i] = -1
            else:
                trend_direction.iloc[i] = trend_direction.iloc[i - 1]

        qqe_trend_line = pd.Series(index=source.index)
        for i in range(len(source)):
            if trend_direction.iloc[i] == 1:
                qqe_trend_line.iloc[i] = long_band.iloc[i]
            else:
                qqe_trend_line.iloc[i] = short_band.iloc[i]

        return qqe_trend_line, smoothed_rsi

    # Primary QQE Calculations
    primary_qqe_trend_line, primary_rsi = calculate_qqe(rsi_length_primary, rsi_smoothing_primary,
                                                          qqe_factor_primary, df['close'])

    # Secondary QQE Calculations
    secondary_qqe_trend_line, secondary_rsi = calculate_qqe(rsi_length_secondary, rsi_smoothing_secondary,
                                                              qqe_factor_secondary, df['close'])

    # Adjustments
    primary_qqe_trend_line_adj = primary_qqe_trend_line - 50
    bollinger_basis = primary_qqe_trend_line_adj.rolling(window=bollinger_length).mean()
    bollinger_deviation = bollinger_multiplier * primary_qqe_trend_line_adj.rolling(window=bollinger_length).std()
    bollinger_upper = bollinger_basis + bollinger_deviation
    bollinger_lower = bollinger_basis - bollinger_deviation

    primary_rsi_adj = primary_rsi - 50
    secondary_rsi_adj = secondary_rsi - 50

    # Add calculated values to the DataFrame
    df['Primary RSI'] = primary_rsi_adj
    df['Secondary RSI'] = secondary_rsi_adj
    df['Bollinger Upper'] = bollinger_upper
    df['Bollinger Lower'] = bollinger_lower

    # Generate trading signals
    df['QQE_Signal'] = np.where((secondary_rsi_adj > threshold_secondary) & (primary_rsi_adj > bollinger_upper),
                                'Up',
                                np.where((secondary_rsi_adj < -threshold_secondary) & (primary_rsi_adj < bollinger_lower),
                                         'Down', None))

    return df

def add_ssl_channel_color(df, atr_period=14, atr_mult=1, atr_smoothing='WMA', baseline_length=60):
    """
    Adds an "SSL_Channel_Color" column to the given DataFrame based on SSL Channel logic.

    Parameters:
        df (pd.DataFrame): Input DataFrame with 'high', 'low', and 'close' columns.
        atr_period (int): Period for ATR calculation.
        atr_mult (float): Multiplier for ATR bands.
        atr_smoothing (str): Smoothing type for ATR ('WMA' or 'SMA').
        baseline_length (int): Length for the baseline HMA calculation.

    Returns:
        pd.DataFrame: Updated DataFrame with "SSL_Channel_Color" column.
    """
    def wma(series, length):
        """Weighted Moving Average (WMA)"""
        weights = np.arange(1, length + 1)
        return series.rolling(length).apply(lambda prices: np.dot(prices, weights) / weights.sum(), raw=True)

    def hma(series, length):
        """Hull Moving Average (HMA)"""
        half_length = int(length / 2)
        sqrt_length = int(np.sqrt(length))
        wma1 = wma(series, half_length)
        wma2 = wma(series, length)
        diff = 2 * wma1 - wma2
        return wma(diff, sqrt_length)

    def calculate_atr(data, length=14, smoothing='WMA'):
        """Average True Range (ATR) calculation"""
        high = data['high']
        low = data['low']
        close = data['close']
        
        tr1 = high - low
        tr2 = abs(high - close.shift(1))
        tr3 = abs(low - close.shift(1))
        tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)

        if smoothing == 'WMA':
            atr = wma(tr, length)
        else:
            # Default to simple moving average if not WMA
            atr = tr.rolling(window=length).mean()
        return atr

    # Ensure no NaN values and reset index
    df.dropna(inplace=True)
    df.reset_index(drop=True, inplace=True)

    # Baseline calculation using HMA
    baseline_hma = hma(df['close'], baseline_length)

    # ATR calculation
    atr_values = calculate_atr(df, length=atr_period, smoothing=atr_smoothing)
    upper_band = df['close'] + atr_values * atr_mult
    lower_band = df['close'] - atr_values * atr_mult

    # SSL Channel calculations
    ema_high = hma(df['high'], baseline_length)
    ema_low = hma(df['low'], baseline_length)

    hlv = np.where(df['close'] > ema_high, 1, np.where(df['close'] < ema_low, -1, np.nan))
    hlv = pd.Series(hlv).ffill()
    
    ssl_down = np.where(hlv < 0, ema_high, ema_low)
    ssl_down_series = pd.Series(ssl_down, index=df.index)

    # Determine SSL Channel Color
    ssl_channel_color = np.where(df['close'] > ssl_down_series, 'blue', 
                                 np.where(df['close'] < ssl_down_series, 'red', np.nan))
    
    # Add SSL_Channel_Color to DataFrame
    df['SSL_Channel_Color'] = ssl_channel_color

    return df

def calculate_rma(df, length, column='close'):
    alpha = 1 / length
    rma = pd.Series(index=df.index, dtype=float)
    rma.iloc[length-1] = df[column].iloc[:length].mean()
    
    for i in range(length, len(df)):
        rma.iloc[i] = (df[column].iloc[i] * alpha) + (rma.iloc[i-1] * (1 - alpha))
    return rma

def calculate_atr(data, period=10):
    data['TR1'] = data['high'] - data['low']
    data['TR2'] = abs(data['high'] - data['close'].shift(1))
    data['TR3'] = abs(data['low'] - data['close'].shift(1))
    data['TR'] = data[['TR1', 'TR2', 'TR3']].max(axis=1)
    data['ATR'] = calculate_rma(data, length=period, column='TR')
    return data

def calculate_supertrend(df, period=10, multiplier=3.0):
    df = calculate_atr(df, period=period)
    high, low, close, atr = df['high'], df['low'], df['close'], df['ATR']
    
    hl2 = (high + low) / 2
    final_upperband = hl2 - (multiplier * atr)
    final_lowerband = hl2 + (multiplier * atr)

    supertrend = pd.DataFrame(index=df.index)
    supertrend['up'] = final_upperband
    supertrend['down'] = final_lowerband
    supertrend['trend'] = 1
    
    for i in range(1, len(df.index)):
        curr, prev = df.index[i], df.index[i-1]
        
        if close[curr] > supertrend.loc[prev, 'up']:
            supertrend.loc[curr, 'up'] = max(supertrend.loc[prev, 'up'], final_upperband[curr])
        else:
            supertrend.loc[curr, 'up'] = final_upperband[curr]
            
        if close[curr] < supertrend.loc[prev, 'down']:
            supertrend.loc[curr, 'down'] = min(supertrend.loc[prev, 'down'], final_lowerband[curr])
        else:
            supertrend.loc[curr, 'down'] = final_lowerband[curr]
            
        supertrend.loc[curr, 'trend'] = 1 if close[curr] > supertrend.loc[prev, 'down'] else \
                                        -1 if close[curr] < supertrend.loc[prev, 'up'] else \
                                        supertrend.loc[prev, 'trend']
            
    supertrend['buy_signal'] = (supertrend['trend'] == 1) & (supertrend['trend'].shift(1) == -1)
    supertrend['sell_signal'] = (supertrend['trend'] == -1) & (supertrend['trend'].shift(1) == 1)
    
    return supertrend

def apply_supertrend(hist, period=10, multiplier=3.0):
    st = calculate_supertrend(hist, period=period, multiplier=multiplier)
    
    hist['SuperTrend_Up'] = np.where(st['trend'] == 1, st['up'], np.nan)
    hist['SuperTrend_Down'] = np.where(st['trend'] == -1, st['down'], np.nan)
    hist['Trend'] = st['trend']
    hist['Buy_Signal'] = st['buy_signal']
    hist['Sell_Signal'] = st['sell_signal']

    hist.drop(columns=['TR1', 'TR2', 'TR3', 'TR'], inplace=True)
    
    return hist

def volume_heatmap(df, length=60, slength=60,
                   threshold_extra_high=4.0,
                   threshold_high=2.5,
                   threshold_medium=1.0,
                   threshold_normal=-0.5):
    """
    Classify volume based on its deviation from the moving average
    """
    # Calculate moving average of volume
    mean = df['volume'].rolling(window=length).mean()

    # Calculate standard deviation of volume
    std = df['volume'].rolling(window=slength).std()

    # Calculate how many standard deviations volume is from mean
    stdbar = (df['volume'] - mean) / std
    df['stdbar'] = stdbar

    # Determine if price closed up or down
    direction = df['close'] > df['open']

    # Classify volume levels
    def classify_volume(row):
        if pd.isna(row['stdbar']):
            return 'Insufficient Data'

        if row['stdbar'] > threshold_extra_high:
            return 'Extra High Volume' + (' Up' if row['direction'] else ' Down')
        elif row['stdbar'] > threshold_high:
            return 'High Volume' + (' Up' if row['direction'] else ' Down')
        elif row['stdbar'] > threshold_medium:
            return 'Medium Volume' + (' Up' if row['direction'] else ' Down')
        elif row['stdbar'] > threshold_normal:
            return 'Normal Volume' + (' Up' if row['direction'] else ' Down')
        else:
            return 'Low Volume' + (' Up' if row['direction'] else ' Down')

    temp_df = pd.DataFrame({
        'stdbar': stdbar,
        'direction': direction
    })

    return temp_df.apply(classify_volume, axis=1)

async def check_and_trade():
    while True:
        # Fetch current positions
        positions = ib.positions()
        position_sizes = {pos.contract.symbol: pos.position for pos in positions}

        # Fetch open orders
        open_orders = ib.reqOpenOrders()
        order_symbols = {order.contract.symbol for order in open_orders}

        for contract in contracts:
            symbol = contract.symbol
            print(f"evaluating  contract for {symbol} ")

            # Fetch historical data
            bars = ib.reqHistoricalData(
                contract,
                endDateTime='',
                durationStr='2 D',
                barSizeSetting='5 mins',
                whatToShow='TRADES',
                useRTH=True,
                formatDate=1
            )

            if not bars:
                print(f"No historical data for {symbol}")
                continue

            # Convert to DataFrame
            df = util.df(bars)
            
            # Apply technical indicators
            df = add_ssl_channel_color(df)
            df['Volume_Label'] = volume_heatmap(df)
            df = calculate_qqe_signal(df)
            df = apply_supertrend(df)

            current_label = df['Volume_Label'].iloc[-1]
            current_row = df.iloc[-1]
            current_position = position_sizes.get(symbol, 0)
            has_pending_order = symbol in order_symbols
            current_price = current_row['close']

            # Check for open trades
            open_trade = trades_collection.find_one({
                'instrument': symbol,
                'status': 'open'
            })

            # Position management
            if open_trade:
                profit_loss_per_share = current_price - open_trade['entry_price']
                current_pnl = profit_loss_per_share * open_trade['quantity']

                # Update current PnL
                update_trade_document(open_trade['_id'], {
                    'current_pnl': current_pnl
                })

                # Take profit at $1 per share
                if profit_loss_per_share >= 1.0:
                    order = MarketOrder('SELL', open_trade['quantity'])
                    trade = ib.placeOrder(contract, order)
                    
                    # Update trade document
                    update_trade_document(open_trade['_id'], {
                        'exit_price': current_price,
                        'exit_timestamp': datetime.now(),
                        'reason': 'target_hit',
                        'realized_pnl': current_pnl,
                        'status': 'closed'
                    })
                    
                    print(f"Take profit executed for {symbol} at {current_price}")

                # Stop loss at -$1 per share
                elif profit_loss_per_share <= -1.0:
                    order = MarketOrder('SELL', open_trade['quantity'])
                    trade = ib.placeOrder(contract, order)
                    
                    # Update trade document
                    update_trade_document(open_trade['_id'], {
                        'exit_price': current_price,
                        'exit_timestamp': datetime.now(),
                        'reason': 'stop_loss',
                        'realized_pnl': current_pnl,
                        'status': 'closed'
                    })
                    
                    print(f"Stop loss executed for {symbol} at {current_price}")

            # Entry management
            else:
                # Check for buy signal
                if (current_row['SSL_Channel_Color'] == 'blue' and 
                    current_row['Trend'] == 1 and 
                    current_row['QQE_Signal'] == 'Up' and 
                    not has_pending_order):
                    
                    order_size = 10  # Adjust as needed
                    order = MarketOrder('BUY', order_size)
                    #print(f"placing BUY order for {symbol} ")
                    trade = ib.placeOrder(contract, order)

                    # Create new trade document
                    trade_doc = create_trade_document(
                        symbol=symbol,
                        direction='long',
                        entry_price=current_price,
                        quantity=order_size
                    )
                    trades_collection.insert_one(trade_doc)

                    print(f"New long position opened for {symbol} at {current_price}")

            # Sleep briefly to avoid pacing violations
            await asyncio.sleep(1)

        # Wait for 5 minutes before next iteration
        print("Waiting for 5 minutes...")
        await asyncio.sleep(300)

# Run the async loop
ib.run(check_and_trade())

# implementation with trailing stop loss and one trade one day only - once ; 

In [ ]:
import asyncio
from ib_insync import *
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
from pymongo import MongoClient
from bson import ObjectId

# Connect to MongoDB
client = MongoClient('mongodb://localhost:27017/')
db = client['trading_db']
trades_collection = db['trades_v2']
exclude_tickers_collection = db['excluded_tickers_v2']  # New collection to track tickers

# Connect to IB
util.startLoop()
ib = IB()
ib.connect('127.0.0.1', 7497, clientId=random.randint(1, 1000))  # Update port and clientId if necessary

tickers = ['AAPL', 'TSLA', 'AMZN', 'GOOG', 'COIN', 'NVDA', 'BA','TSM']
contracts = [Stock(ticker, 'SMART', 'USD') for ticker in tickers]

# Ensure contracts are resolved
ib.qualifyContracts(*contracts)

# Helper function to create trade document
def create_trade_document(symbol, direction, entry_price, quantity):
    return {
        'instrument': symbol,
        'direction': direction,
        'entry_price': entry_price,
        'highest_price': entry_price,  # Initialize the highest price with the entry price
        'quantity': quantity,
        'entry_timestamp': datetime.now(),
        'order_id': str(ObjectId()),
        'exit_price': None,
        'exit_timestamp': None,
        'reason': None,
        'current_pnl': 0,
        'realized_pnl': 0,
        'status': 'open',
        'created_at': datetime.now(),
        'updated_at': datetime.now()
    }

# Helper function to update trade document
def update_trade_document(trade_id, updates):
    updates['updated_at'] = datetime.now()
    trades_collection.update_one(
        {'_id': ObjectId(trade_id)},
        {'$set': updates}
    )

# Include your indicator functions here (calculate_qqe_signal, add_ssl_channel_color, apply_supertrend, volume_heatmap)
def calculate_qqe_signal(df,
                         rsi_length_primary=6,
                         rsi_smoothing_primary=5,
                         qqe_factor_primary=3.0,
                         threshold_primary=3.0,
                         rsi_length_secondary=6,
                         rsi_smoothing_secondary=5,
                         qqe_factor_secondary=1.61,
                         threshold_secondary=3.0,
                         bollinger_length=50,
                         bollinger_multiplier=0.35):
    """
    Calculates the QQE (Quantitative Qualitative Estimation) signal and adds it as a column to the DataFrame.

    Parameters:
        df (pd.DataFrame): Input DataFrame with a 'close' column.
        rsi_length_primary (int): RSI length for the primary QQE.
        rsi_smoothing_primary (int): Smoothing factor for the primary RSI.
        qqe_factor_primary (float): QQE factor for the primary QQE.
        threshold_primary (float): Threshold for the primary QQE.
        rsi_length_secondary (int): RSI length for the secondary QQE.
        rsi_smoothing_secondary (int): Smoothing factor for the secondary RSI.
        qqe_factor_secondary (float): QQE factor for the secondary QQE.
        threshold_secondary (float): Threshold for the secondary QQE.
        bollinger_length (int): Length for the Bollinger Bands.
        bollinger_multiplier (float): Multiplier for the Bollinger Bands.

    Returns:
        pd.DataFrame: DataFrame with added columns 'Primary RSI', 'Secondary RSI', 'Bollinger Upper', 'Bollinger Lower', and 'QQE_Signal'.
    """

    def calculate_qqe(rsi_length, smoothing_factor, qqe_factor, source):
        wilders_length = rsi_length * 2 - 1

        delta = source.diff()
        up = delta.clip(lower=0)
        down = -delta.clip(upper=0)
        up_avg = up.ewm(alpha=1 / rsi_length, adjust=False).mean()
        down_avg = down.ewm(alpha=1 / rsi_length, adjust=False).mean()
        rs = up_avg / down_avg
        rsi = 100 - (100 / (1 + rs))

        smoothed_rsi = rsi.ewm(span=smoothing_factor, adjust=False).mean()

        atr_rsi = (smoothed_rsi - smoothed_rsi.shift(1)).abs()
        smoothed_atr_rsi = atr_rsi.ewm(span=wilders_length, adjust=False).mean()
        dynamic_atr_rsi = smoothed_atr_rsi * qqe_factor

        long_band = pd.Series(index=source.index)
        short_band = pd.Series(index=source.index)
        trend_direction = pd.Series(index=source.index, dtype='int')

        long_band.iloc[0] = smoothed_rsi.iloc[0]
        short_band.iloc[0] = smoothed_rsi.iloc[0]
        trend_direction.iloc[0] = 0

        for i in range(1, len(source)):
            atr_delta = dynamic_atr_rsi.iloc[i]
            new_long_band = smoothed_rsi.iloc[i] - atr_delta
            new_short_band = smoothed_rsi.iloc[i] + atr_delta

            prev_long_band = long_band.iloc[i - 1]
            prev_short_band = short_band.iloc[i - 1]
            prev_smoothed_rsi = smoothed_rsi.iloc[i - 1]

            if prev_smoothed_rsi > prev_long_band and smoothed_rsi.iloc[i] > prev_long_band:
                long_band.iloc[i] = max(prev_long_band, new_long_band)
            else:
                long_band.iloc[i] = new_long_band

            if prev_smoothed_rsi < prev_short_band and smoothed_rsi.iloc[i] < prev_short_band:
                short_band.iloc[i] = min(prev_short_band, new_short_band)
            else:
                short_band.iloc[i] = new_short_band

            if smoothed_rsi.iloc[i] > prev_short_band and prev_smoothed_rsi <= prev_short_band:
                trend_direction.iloc[i] = 1
            elif smoothed_rsi.iloc[i] < prev_long_band and prev_smoothed_rsi >= prev_long_band:
                trend_direction.iloc[i] = -1
            else:
                trend_direction.iloc[i] = trend_direction.iloc[i - 1]

        qqe_trend_line = pd.Series(index=source.index)
        for i in range(len(source)):
            if trend_direction.iloc[i] == 1:
                qqe_trend_line.iloc[i] = long_band.iloc[i]
            else:
                qqe_trend_line.iloc[i] = short_band.iloc[i]

        return qqe_trend_line, smoothed_rsi

    # Primary QQE Calculations
    primary_qqe_trend_line, primary_rsi = calculate_qqe(rsi_length_primary, rsi_smoothing_primary,
                                                          qqe_factor_primary, df['close'])

    # Secondary QQE Calculations
    secondary_qqe_trend_line, secondary_rsi = calculate_qqe(rsi_length_secondary, rsi_smoothing_secondary,
                                                              qqe_factor_secondary, df['close'])

    # Adjustments
    primary_qqe_trend_line_adj = primary_qqe_trend_line - 50
    bollinger_basis = primary_qqe_trend_line_adj.rolling(window=bollinger_length).mean()
    bollinger_deviation = bollinger_multiplier * primary_qqe_trend_line_adj.rolling(window=bollinger_length).std()
    bollinger_upper = bollinger_basis + bollinger_deviation
    bollinger_lower = bollinger_basis - bollinger_deviation

    primary_rsi_adj = primary_rsi - 50
    secondary_rsi_adj = secondary_rsi - 50

    # Add calculated values to the DataFrame
    df['Primary RSI'] = primary_rsi_adj
    df['Secondary RSI'] = secondary_rsi_adj
    df['Bollinger Upper'] = bollinger_upper
    df['Bollinger Lower'] = bollinger_lower

    # Generate trading signals
    df['QQE_Signal'] = np.where((secondary_rsi_adj > threshold_secondary) & (primary_rsi_adj > bollinger_upper),
                                'Up',
                                np.where((secondary_rsi_adj < -threshold_secondary) & (primary_rsi_adj < bollinger_lower),
                                         'Down', None))

    return df

def add_ssl_channel_color(df, atr_period=14, atr_mult=1, atr_smoothing='WMA', baseline_length=60):
    """
    Adds an "SSL_Channel_Color" column to the given DataFrame based on SSL Channel logic.

    Parameters:
        df (pd.DataFrame): Input DataFrame with 'high', 'low', and 'close' columns.
        atr_period (int): Period for ATR calculation.
        atr_mult (float): Multiplier for ATR bands.
        atr_smoothing (str): Smoothing type for ATR ('WMA' or 'SMA').
        baseline_length (int): Length for the baseline HMA calculation.

    Returns:
        pd.DataFrame: Updated DataFrame with "SSL_Channel_Color" column.
    """
    def wma(series, length):
        """Weighted Moving Average (WMA)"""
        weights = np.arange(1, length + 1)
        return series.rolling(length).apply(lambda prices: np.dot(prices, weights) / weights.sum(), raw=True)

    def hma(series, length):
        """Hull Moving Average (HMA)"""
        half_length = int(length / 2)
        sqrt_length = int(np.sqrt(length))
        wma1 = wma(series, half_length)
        wma2 = wma(series, length)
        diff = 2 * wma1 - wma2
        return wma(diff, sqrt_length)

    def calculate_atr(data, length=14, smoothing='WMA'):
        """Average True Range (ATR) calculation"""
        high = data['high']
        low = data['low']
        close = data['close']
        
        tr1 = high - low
        tr2 = abs(high - close.shift(1))
        tr3 = abs(low - close.shift(1))
        tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)

        if smoothing == 'WMA':
            atr = wma(tr, length)
        else:
            # Default to simple moving average if not WMA
            atr = tr.rolling(window=length).mean()
        return atr

    # Ensure no NaN values and reset index
    df.dropna(inplace=True)
    df.reset_index(drop=True, inplace=True)

    # Baseline calculation using HMA
    baseline_hma = hma(df['close'], baseline_length)

    # ATR calculation
    atr_values = calculate_atr(df, length=atr_period, smoothing=atr_smoothing)
    upper_band = df['close'] + atr_values * atr_mult
    lower_band = df['close'] - atr_values * atr_mult

    # SSL Channel calculations
    ema_high = hma(df['high'], baseline_length)
    ema_low = hma(df['low'], baseline_length)

    hlv = np.where(df['close'] > ema_high, 1, np.where(df['close'] < ema_low, -1, np.nan))
    hlv = pd.Series(hlv).ffill()
    
    ssl_down = np.where(hlv < 0, ema_high, ema_low)
    ssl_down_series = pd.Series(ssl_down, index=df.index)

    # Determine SSL Channel Color
    ssl_channel_color = np.where(df['close'] > ssl_down_series, 'blue', 
                                 np.where(df['close'] < ssl_down_series, 'red', np.nan))
    
    # Add SSL_Channel_Color to DataFrame
    df['SSL_Channel_Color'] = ssl_channel_color

    return df

def calculate_rma(df, length, column='close'):
    alpha = 1 / length
    rma = pd.Series(index=df.index, dtype=float)
    rma.iloc[length-1] = df[column].iloc[:length].mean()
    
    for i in range(length, len(df)):
        rma.iloc[i] = (df[column].iloc[i] * alpha) + (rma.iloc[i-1] * (1 - alpha))
    return rma

def calculate_atr(data, period=10):
    data['TR1'] = data['high'] - data['low']
    data['TR2'] = abs(data['high'] - data['close'].shift(1))
    data['TR3'] = abs(data['low'] - data['close'].shift(1))
    data['TR'] = data[['TR1', 'TR2', 'TR3']].max(axis=1)
    data['ATR'] = calculate_rma(data, length=period, column='TR')
    return data

def calculate_supertrend(df, period=10, multiplier=3.0):
    df = calculate_atr(df, period=period)
    high, low, close, atr = df['high'], df['low'], df['close'], df['ATR']
    
    hl2 = (high + low) / 2
    final_upperband = hl2 - (multiplier * atr)
    final_lowerband = hl2 + (multiplier * atr)

    supertrend = pd.DataFrame(index=df.index)
    supertrend['up'] = final_upperband
    supertrend['down'] = final_lowerband
    supertrend['trend'] = 1
    
    for i in range(1, len(df.index)):
        curr, prev = df.index[i], df.index[i-1]
        
        if close[curr] > supertrend.loc[prev, 'up']:
            supertrend.loc[curr, 'up'] = max(supertrend.loc[prev, 'up'], final_upperband[curr])
        else:
            supertrend.loc[curr, 'up'] = final_upperband[curr]
            
        if close[curr] < supertrend.loc[prev, 'down']:
            supertrend.loc[curr, 'down'] = min(supertrend.loc[prev, 'down'], final_lowerband[curr])
        else:
            supertrend.loc[curr, 'down'] = final_lowerband[curr]
            
        supertrend.loc[curr, 'trend'] = 1 if close[curr] > supertrend.loc[prev, 'down'] else \
                                        -1 if close[curr] < supertrend.loc[prev, 'up'] else \
                                        supertrend.loc[prev, 'trend']
            
    supertrend['buy_signal'] = (supertrend['trend'] == 1) & (supertrend['trend'].shift(1) == -1)
    supertrend['sell_signal'] = (supertrend['trend'] == -1) & (supertrend['trend'].shift(1) == 1)
    
    return supertrend

def apply_supertrend(hist, period=10, multiplier=3.0):
    st = calculate_supertrend(hist, period=period, multiplier=multiplier)
    
    hist['SuperTrend_Up'] = np.where(st['trend'] == 1, st['up'], np.nan)
    hist['SuperTrend_Down'] = np.where(st['trend'] == -1, st['down'], np.nan)
    hist['Trend'] = st['trend']
    hist['Buy_Signal'] = st['buy_signal']
    hist['Sell_Signal'] = st['sell_signal']

    hist.drop(columns=['TR1', 'TR2', 'TR3', 'TR'], inplace=True)
    
    return hist

def volume_heatmap(df, length=60, slength=60,
                   threshold_extra_high=4.0,
                   threshold_high=2.5,
                   threshold_medium=1.0,
                   threshold_normal=-0.5):
    """
    Classify volume based on its deviation from the moving average
    """
    # Calculate moving average of volume
    mean = df['volume'].rolling(window=length).mean()

    # Calculate standard deviation of volume
    std = df['volume'].rolling(window=slength).std()

    # Calculate how many standard deviations volume is from mean
    stdbar = (df['volume'] - mean) / std
    df['stdbar'] = stdbar

    # Determine if price closed up or down
    direction = df['close'] > df['open']

    # Classify volume levels
    def classify_volume(row):
        if pd.isna(row['stdbar']):
            return 'Insufficient Data'

        if row['stdbar'] > threshold_extra_high:
            return 'Extra High Volume' + (' Up' if row['direction'] else ' Down')
        elif row['stdbar'] > threshold_high:
            return 'High Volume' + (' Up' if row['direction'] else ' Down')
        elif row['stdbar'] > threshold_medium:
            return 'Medium Volume' + (' Up' if row['direction'] else ' Down')
        elif row['stdbar'] > threshold_normal:
            return 'Normal Volume' + (' Up' if row['direction'] else ' Down')
        else:
            return 'Low Volume' + (' Up' if row['direction'] else ' Down')

    temp_df = pd.DataFrame({
        'stdbar': stdbar,
        'direction': direction
    })

    return temp_df.apply(classify_volume, axis=1)

async def check_and_trade():
    trailing_amount = 1.0  # Trailing stop amount in dollars

    while True:
        # Fetch current positions
        positions = ib.positions()
        position_sizes = {pos.contract.symbol: pos.position for pos in positions}

        # Fetch open orders
        open_orders = ib.reqOpenOrders()
        order_symbols = {order.contract.symbol for order in open_orders}

        for contract in contracts:
            symbol = contract.symbol
            print(f"Evaluating contract for {symbol}")

            # Fetch historical data
            bars = ib.reqHistoricalData(
                contract,
                endDateTime='',
                durationStr='2 D',
                barSizeSetting='5 mins',
                whatToShow='TRADES',
                useRTH=True,
                formatDate=1
            )

            if not bars:
                print(f"No historical data for {symbol}")
                continue

            # Convert to DataFrame
            df = util.df(bars)

            # Apply technical indicators
            df = add_ssl_channel_color(df)
            df['Volume_Label'] = volume_heatmap(df)
            df = calculate_qqe_signal(df)
            df = apply_supertrend(df)

            current_label = df['Volume_Label'].iloc[-1]
            current_row = df.iloc[-1]
            current_position = position_sizes.get(symbol, 0)
            has_pending_order = symbol in order_symbols
            current_price = current_row['close']

            # Check for open trades
            open_trade = trades_collection.find_one({
                'instrument': symbol,
                'status': 'open'
            })

            # Get current date
            current_date = datetime.now().date()

            # Position management
            if open_trade:
                entry_price = open_trade['entry_price']
                highest_price = open_trade.get('highest_price', entry_price)

                # Update highest price if current price is higher
                if current_price > highest_price:
                    highest_price = current_price
                    update_trade_document(open_trade['_id'], {
                        'highest_price': highest_price
                    })

                # Calculate trailing stop price
                trailing_stop_price = highest_price - trailing_amount
                profit_loss_per_share = current_price - entry_price
                current_pnl = profit_loss_per_share * open_trade['quantity']

                # Update current PnL
                update_trade_document(open_trade['_id'], {
                    'current_pnl': current_pnl
                })

                # Check if current price has fallen below trailing stop price
                if current_price <= trailing_stop_price:
                    order = MarketOrder('SELL', open_trade['quantity'])
                    trade = ib.placeOrder(contract, order)

                    # Update trade document
                    update_trade_document(open_trade['_id'], {
                        'exit_price': current_price,
                        'exit_timestamp': datetime.now(),
                        'reason': 'trailing_stop_hit',
                        'realized_pnl': current_pnl,
                        'status': 'closed'
                    })

                    print(f"Trailing stop loss executed for {symbol} at {current_price}")

                # Optional: Take profit at $1 per share (if desired)
                elif profit_loss_per_share >= 1.0:
                    order = MarketOrder('SELL', open_trade['quantity'])
                    trade = ib.placeOrder(contract, order)

                    # Update trade document
                    update_trade_document(open_trade['_id'], {
                        'exit_price': current_price,
                        'exit_timestamp': datetime.now(),
                        'reason': 'target_hit',
                        'realized_pnl': current_pnl,
                        'status': 'closed'
                    })

                    print(f"Take profit executed for {symbol} at {current_price}")

            # Entry management
            else:
                # Check if we have already traded this ticker today
                exclude_entry = exclude_tickers_collection.find_one({
                    'ticker': symbol,
                    'date': current_date.isoformat()
                })

                if exclude_entry:
                    print(f"Already traded {symbol} today. Skipping new trade.")
                    continue  # Skip to next ticker

                # Check for buy signal
                if (current_row['SSL_Channel_Color'] == 'blue' and
                    current_row['Trend'] == 1 and
                    current_row['QQE_Signal'] == 'Up' and
                    not has_pending_order):

                    order_size = 10  # Adjust as needed
                    order = MarketOrder('BUY', order_size)
                    trade = ib.placeOrder(contract, order)

                    # Create new trade document with highest_price initialized to entry_price
                    trade_doc = create_trade_document(
                        symbol=symbol,
                        direction='long',
                        entry_price=current_price,
                        quantity=order_size
                    )
                    trades_collection.insert_one(trade_doc)

                    # Add ticker to exclude_tickers collection
                    exclude_tickers_collection.insert_one({
                        'ticker': symbol,
                        'date': current_date.isoformat()
                    })

                    print(f"New long position opened for {symbol} at {current_price}")

            # Sleep briefly to avoid pacing violations
            await asyncio.sleep(1)

        # Wait for 5 minutes before next iteration
        print("Waiting for 5 minutes...")
        await asyncio.sleep(300)

# Run the async loop
try:
    ib.run(check_and_trade())
except Exception as e:
    print(f"An error occurred: {e}")
finally:
    ib.disconnect()
    print("Disconnected from IB.")

## NEW CODE TO MAXIMIZE PROFIT ( older version )

In [ ]:
import asyncio
from ib_insync import *
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
from pymongo import MongoClient
from bson import ObjectId
import warnings

# Method 1: Suppress all warnings
warnings.filterwarnings("ignore")

# Method 2: Suppress specific warning types
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Connect to MongoDB
client = MongoClient('mongodb://localhost:27017/')
db = client['trading_db']
trades_collection = db['trades_v2']
exclude_tickers_collection = db['excluded_tickers_v2']  # New collection to track tickers

# Connect to IB
util.startLoop()
ib = IB()
ib.connect('127.0.0.1', 7497, clientId=random.randint(1, 1000))  # Update port and clientId if necessary

tickers =  ['TSLA', 'COIN' ] #['AAPL', 'TSLA', 'AMZN', 'GOOG', 'COIN', 'NVDA', 'BA','TSM']
contracts = [Stock(ticker, 'SMART', 'USD') for ticker in tickers]

# Ensure contracts are resolved
ib.qualifyContracts(*contracts)

# Update the create_trade_document function to include the new fields
def create_trade_document(symbol, direction, entry_price, quantity):
    return {
        'instrument': symbol,
        'direction': direction,
        'entry_price': entry_price,
        'highest_price': entry_price,  # Initialize the highest price with the entry price
        'quantity': quantity,
        'current_quantity': quantity,  # New field to track remaining quantity
        'partial_exits': [],           # New field to track partial exit information
        'entry_timestamp': datetime.now(),
        'order_id': str(ObjectId()),
        'exit_price': None,
        'exit_timestamp': None,
        'reason': None,
        'current_pnl': 0,
        'realized_pnl': 0,
        'status': 'open',
        'created_at': datetime.now(),
        'updated_at': datetime.now()
    }

# Helper function to update trade document
def update_trade_document(trade_id, updates):
    updates['updated_at'] = datetime.now()
    trades_collection.update_one(
        {'_id': ObjectId(trade_id)},
        {'$set': updates}
    )

async def check_and_trade():
    # Configuration parameters for position management
    base_trailing_amount = 1.0  # Base trailing stop amount in dollars
    min_trailing_pct = 0.005  # Minimum trailing stop as percentage of price (0.5%)
    atr_multiplier = 1.5  # ATR multiplier for dynamic trailing stops
    
    # Partial profit-taking levels
    profit_level_1 = 0.01  # 1% profit
    profit_level_2 = 0.02  # 2% profit
    profit_level_3 = 0.03  # 3% profit
    
    # Partial position sizing for exits
    partial_size_1 = 0.25  # Exit 25% at first profit level
    partial_size_2 = 0.25  # Exit 25% at second profit level
    partial_size_3 = 0.25  # Exit 25% at third profit level
    # Remaining 25% follows trailing stop
    
    while True:
        # Fetch current positions
        positions = ib.positions()
        position_sizes = {pos.contract.symbol: pos.position for pos in positions}

        # Fetch open orders
        open_orders = ib.reqOpenOrders()
        order_symbols = {order.contract.symbol for order in open_orders}

        for contract in contracts:
            symbol = contract.symbol
            print(f"Evaluating contract for {symbol}")

            # Fetch historical data
            bars = ib.reqHistoricalData(
                contract,
                endDateTime='',
                durationStr='2 D',
                barSizeSetting='5 mins',
                whatToShow='TRADES',
                useRTH=False,
                formatDate=1
            )

            if not bars:
                print(f"No historical data for {symbol}")
                continue

            # Convert to DataFrame
            df = util.df(bars)

            # Apply technical indicators
            df = add_ssl_channel_color(df)
            df['Volume_Label'] = volume_heatmap(df)
            df = calculate_qqe_signal(df)
            df = apply_supertrend(df)
            df = large_cap_trend_meter2(df)
            
            # Calculate ATR for dynamic trailing stop
            if 'ATR' not in df.columns:
                df = calculate_atr(df, period=14)  # Use 14-period ATR

            current_row = df.iloc[-1]
            current_price = current_row['close']
            current_atr = current_row.get('ATR', base_trailing_amount)  # Fallback if ATR calculation fails
            
            # Check for open trades
            open_trade = trades_collection.find_one({
                'instrument': symbol,
                'status': 'open'
            })

            # Get current date
            current_date = datetime.now().date()

            # Position management
            if open_trade:
                entry_price = open_trade['entry_price']
                highest_price = open_trade.get('highest_price', entry_price)
                original_quantity = open_trade['quantity']
                current_quantity = open_trade.get('current_quantity', original_quantity)
                partial_exits = open_trade.get('partial_exits', [])
                time_in_trade = (datetime.now() - open_trade['entry_timestamp']).total_seconds() / 3600  # Hours
                
                # Update highest price if current price is higher
                if current_price > highest_price:
                    highest_price = current_price
                    update_trade_document(open_trade['_id'], {
                        'highest_price': highest_price
                    })

                # Calculate profit as a percentage
                profit_pct = (current_price - entry_price) / entry_price
                profit_loss_per_share = current_price - entry_price
                current_pnl = profit_loss_per_share * current_quantity
                
                # Update current PnL
                update_trade_document(open_trade['_id'], {
                    'current_pnl': current_pnl
                })
                
                # Determine dynamic trailing stop
                # Use max of different methods to create adaptive trailing stop
                atr_based_stop = highest_price - (current_atr * atr_multiplier)
                percentage_based_stop = highest_price * (1 - min_trailing_pct)
                fixed_based_stop = highest_price - base_trailing_amount
                
                # Tighten trail the longer we're in the trade
                time_factor = min(1.0, time_in_trade / 24)  # Full tightening after 24 hours
                trailing_stop_price = atr_based_stop * (1 - 0.25 * time_factor)  # Tighten by up to 25%
                
                # Use the tightest (highest) of the calculated stops
                trailing_stop_price = max(trailing_stop_price, percentage_based_stop, fixed_based_stop)
                
                # Adjust trailing stop based on trend strength
                if current_row['Trend'] == 1 and current_row['QQE_Signal'] == 'Up':
                    # Strong uptrend, give more room
                    trailing_stop_price = trailing_stop_price * 0.98  # 2% more room
                
                # Take partial profits at different levels
                if profit_pct >= profit_level_3 and not any(exit['level'] == 3 for exit in partial_exits):
                    # Take third partial profit
                    exit_quantity = int(original_quantity * partial_size_3)
                    if exit_quantity > 0 and current_quantity >= exit_quantity:
                        order = MarketOrder('SELL', exit_quantity)
                        trade = ib.placeOrder(contract, order)
                        
                        partial_exits.append({
                            'level': 3,
                            'price': current_price,
                            'quantity': exit_quantity,
                            'timestamp': datetime.now()
                        })
                        
                        current_quantity -= exit_quantity
                        update_trade_document(open_trade['_id'], {
                            'current_quantity': current_quantity,
                            'partial_exits': partial_exits
                        })
                        
                        print(f"Third partial profit taken for {symbol} at {current_price} ({profit_pct:.2%})")
                
                elif profit_pct >= profit_level_2 and not any(exit['level'] == 2 for exit in partial_exits):
                    # Take second partial profit
                    exit_quantity = int(original_quantity * partial_size_2)
                    if exit_quantity > 0 and current_quantity >= exit_quantity:
                        order = MarketOrder('SELL', exit_quantity)
                        trade = ib.placeOrder(contract, order)
                        
                        partial_exits.append({
                            'level': 2,
                            'price': current_price,
                            'quantity': exit_quantity,
                            'timestamp': datetime.now()
                        })
                        
                        current_quantity -= exit_quantity
                        update_trade_document(open_trade['_id'], {
                            'current_quantity': current_quantity,
                            'partial_exits': partial_exits
                        })
                        
                        print(f"Second partial profit taken for {symbol} at {current_price} ({profit_pct:.2%})")
                
                elif profit_pct >= profit_level_1 and not any(exit['level'] == 1 for exit in partial_exits):
                    # Take first partial profit
                    exit_quantity = int(original_quantity * partial_size_1)
                    if exit_quantity > 0 and current_quantity >= exit_quantity:
                        order = MarketOrder('SELL', exit_quantity)
                        trade = ib.placeOrder(contract, order)
                        
                        partial_exits.append({
                            'level': 1,
                            'price': current_price,
                            'quantity': exit_quantity,
                            'timestamp': datetime.now()
                        })
                        
                        current_quantity -= exit_quantity
                        update_trade_document(open_trade['_id'], {
                            'current_quantity': current_quantity,
                            'partial_exits': partial_exits
                        })
                        
                        print(f"First partial profit taken for {symbol} at {current_price} ({profit_pct:.2%})")

                # Check for trailing stop exit with remaining position
                if current_price <= trailing_stop_price and current_quantity > 0:
                    order = MarketOrder('SELL', current_quantity)
                    trade = ib.placeOrder(contract, order)

                    # Calculate average exit price considering partial exits
                    total_value = current_price * current_quantity
                    total_quantity = current_quantity
                    
                    for exit in partial_exits:
                        total_value += exit['price'] * exit['quantity']
                        total_quantity += exit['quantity']
                    
                    avg_exit_price = total_value / total_quantity if total_quantity > 0 else current_price
                    
                    # Calculate final P&L
                    realized_pnl = (avg_exit_price - entry_price) * original_quantity

                    # Update trade document
                    update_trade_document(open_trade['_id'], {
                        'exit_price': avg_exit_price,
                        'exit_timestamp': datetime.now(),
                        'reason': 'trailing_stop_hit',
                        'realized_pnl': realized_pnl,
                        'status': 'closed',
                        'current_quantity': 0
                    })

                    print(f"Trailing stop loss executed for {symbol} at {current_price}, avg exit: {avg_exit_price}")

            # Entry management (same as before)
            else:
                # Check if we have already traded this ticker today
                exclude_entry = exclude_tickers_collection.find_one({
                    'ticker': symbol,
                    'date': current_date.isoformat()
                })

                if exclude_entry:
                    print(f"Already traded {symbol} today. Skipping new trade.")
                    continue  # Skip to next ticker

                # Check for buy signal
                if (current_row['SSL_Channel_Color'] == 'blue' and
                    #current_row['Trend'] == 1 and
                    #current_row['QQE_Signal'] == 'Up'
                    current_row['trend_color'] == 'forestgreen' ):

                    order_size = 20  # Adjust as needed
                    order = MarketOrder('BUY', order_size)
                    trade = ib.placeOrder(contract, order)

                    # Create new trade document with additional fields
                    trade_doc = create_trade_document(
                        symbol=symbol,
                        direction='long',
                        entry_price=current_price,
                        quantity=order_size
                    )
                    
                    # Add new fields for the improved trailing stop system
                    trade_doc['current_quantity'] = order_size
                    trade_doc['partial_exits'] = []
                    
                    trades_collection.insert_one(trade_doc)

                    # Add ticker to exclude_tickers collection
                    exclude_tickers_collection.insert_one({
                        'ticker': symbol,
                        'date': current_date.isoformat()
                    })

                    print(f"New long position opened for {symbol} at {current_price}")

            # Sleep briefly to avoid pacing violations
            await asyncio.sleep(1)

        # Wait for 5 minutes before next iteration
        print("Waiting for 5 minutes...")
        await asyncio.sleep(300)

# Run the async loop
try:
    ib.run(check_and_trade())
except Exception as e:
    print(f"An error occurred: {e}")
finally:
    ib.disconnect()
    print("Disconnected from IB.")

# INDICATOR Functions CODE

In [ ]:
# INDICATOR

# Include your indicator functions here (calculate_qqe_signal, add_ssl_channel_color, apply_supertrend, volume_heatmap)
def calculate_qqe_signal(df,
                         rsi_length_primary=6,
                         rsi_smoothing_primary=5,
                         qqe_factor_primary=3.0,
                         threshold_primary=3.0,
                         rsi_length_secondary=6,
                         rsi_smoothing_secondary=5,
                         qqe_factor_secondary=1.61,
                         threshold_secondary=3.0,
                         bollinger_length=50,
                         bollinger_multiplier=0.35):
    """
    Calculates the QQE (Quantitative Qualitative Estimation) signal and adds it as a column to the DataFrame.

    Parameters:
        df (pd.DataFrame): Input DataFrame with a 'close' column.
        rsi_length_primary (int): RSI length for the primary QQE.
        rsi_smoothing_primary (int): Smoothing factor for the primary RSI.
        qqe_factor_primary (float): QQE factor for the primary QQE.
        threshold_primary (float): Threshold for the primary QQE.
        rsi_length_secondary (int): RSI length for the secondary QQE.
        rsi_smoothing_secondary (int): Smoothing factor for the secondary RSI.
        qqe_factor_secondary (float): QQE factor for the secondary QQE.
        threshold_secondary (float): Threshold for the secondary QQE.
        bollinger_length (int): Length for the Bollinger Bands.
        bollinger_multiplier (float): Multiplier for the Bollinger Bands.

    Returns:
        pd.DataFrame: DataFrame with added columns 'Primary RSI', 'Secondary RSI', 'Bollinger Upper', 'Bollinger Lower', and 'QQE_Signal'.
    """

    def calculate_qqe(rsi_length, smoothing_factor, qqe_factor, source):
        wilders_length = rsi_length * 2 - 1

        delta = source.diff()
        up = delta.clip(lower=0)
        down = -delta.clip(upper=0)
        up_avg = up.ewm(alpha=1 / rsi_length, adjust=False).mean()
        down_avg = down.ewm(alpha=1 / rsi_length, adjust=False).mean()
        rs = up_avg / down_avg
        rsi = 100 - (100 / (1 + rs))

        smoothed_rsi = rsi.ewm(span=smoothing_factor, adjust=False).mean()

        atr_rsi = (smoothed_rsi - smoothed_rsi.shift(1)).abs()
        smoothed_atr_rsi = atr_rsi.ewm(span=wilders_length, adjust=False).mean()
        dynamic_atr_rsi = smoothed_atr_rsi * qqe_factor

        long_band = pd.Series(index=source.index)
        short_band = pd.Series(index=source.index)
        trend_direction = pd.Series(index=source.index, dtype='int')

        long_band.iloc[0] = smoothed_rsi.iloc[0]
        short_band.iloc[0] = smoothed_rsi.iloc[0]
        trend_direction.iloc[0] = 0

        for i in range(1, len(source)):
            atr_delta = dynamic_atr_rsi.iloc[i]
            new_long_band = smoothed_rsi.iloc[i] - atr_delta
            new_short_band = smoothed_rsi.iloc[i] + atr_delta

            prev_long_band = long_band.iloc[i - 1]
            prev_short_band = short_band.iloc[i - 1]
            prev_smoothed_rsi = smoothed_rsi.iloc[i - 1]

            if prev_smoothed_rsi > prev_long_band and smoothed_rsi.iloc[i] > prev_long_band:
                long_band.iloc[i] = max(prev_long_band, new_long_band)
            else:
                long_band.iloc[i] = new_long_band

            if prev_smoothed_rsi < prev_short_band and smoothed_rsi.iloc[i] < prev_short_band:
                short_band.iloc[i] = min(prev_short_band, new_short_band)
            else:
                short_band.iloc[i] = new_short_band

            if smoothed_rsi.iloc[i] > prev_short_band and prev_smoothed_rsi <= prev_short_band:
                trend_direction.iloc[i] = 1
            elif smoothed_rsi.iloc[i] < prev_long_band and prev_smoothed_rsi >= prev_long_band:
                trend_direction.iloc[i] = -1
            else:
                trend_direction.iloc[i] = trend_direction.iloc[i - 1]

        qqe_trend_line = pd.Series(index=source.index)
        for i in range(len(source)):
            if trend_direction.iloc[i] == 1:
                qqe_trend_line.iloc[i] = long_band.iloc[i]
            else:
                qqe_trend_line.iloc[i] = short_band.iloc[i]

        return qqe_trend_line, smoothed_rsi

    # Primary QQE Calculations
    primary_qqe_trend_line, primary_rsi = calculate_qqe(rsi_length_primary, rsi_smoothing_primary,
                                                          qqe_factor_primary, df['close'])

    # Secondary QQE Calculations
    secondary_qqe_trend_line, secondary_rsi = calculate_qqe(rsi_length_secondary, rsi_smoothing_secondary,
                                                              qqe_factor_secondary, df['close'])

    # Adjustments
    primary_qqe_trend_line_adj = primary_qqe_trend_line - 50
    bollinger_basis = primary_qqe_trend_line_adj.rolling(window=bollinger_length).mean()
    bollinger_deviation = bollinger_multiplier * primary_qqe_trend_line_adj.rolling(window=bollinger_length).std()
    bollinger_upper = bollinger_basis + bollinger_deviation
    bollinger_lower = bollinger_basis - bollinger_deviation

    primary_rsi_adj = primary_rsi - 50
    secondary_rsi_adj = secondary_rsi - 50

    # Add calculated values to the DataFrame
    df['Primary RSI'] = primary_rsi_adj
    df['Secondary RSI'] = secondary_rsi_adj
    df['Bollinger Upper'] = bollinger_upper
    df['Bollinger Lower'] = bollinger_lower

    # Generate trading signals
    df['QQE_Signal'] = np.where((secondary_rsi_adj > threshold_secondary) & (primary_rsi_adj > bollinger_upper),
                                'Up',
                                np.where((secondary_rsi_adj < -threshold_secondary) & (primary_rsi_adj < bollinger_lower),
                                         'Down', None))

    return df

def add_ssl_channel_color(df, atr_period=14, atr_mult=1, atr_smoothing='WMA', baseline_length=60):
    """
    Adds an "SSL_Channel_Color" column to the given DataFrame based on SSL Channel logic.

    Parameters:
        df (pd.DataFrame): Input DataFrame with 'high', 'low', and 'close' columns.
        atr_period (int): Period for ATR calculation.
        atr_mult (float): Multiplier for ATR bands.
        atr_smoothing (str): Smoothing type for ATR ('WMA' or 'SMA').
        baseline_length (int): Length for the baseline HMA calculation.

    Returns:
        pd.DataFrame: Updated DataFrame with "SSL_Channel_Color" column.
    """
    def wma(series, length):
        """Weighted Moving Average (WMA)"""
        weights = np.arange(1, length + 1)
        return series.rolling(length).apply(lambda prices: np.dot(prices, weights) / weights.sum(), raw=True)

    def hma(series, length):
        """Hull Moving Average (HMA)"""
        half_length = int(length / 2)
        sqrt_length = int(np.sqrt(length))
        wma1 = wma(series, half_length)
        wma2 = wma(series, length)
        diff = 2 * wma1 - wma2
        return wma(diff, sqrt_length)

    def calculate_atr(data, length=14, smoothing='WMA'):
        """Average True Range (ATR) calculation"""
        high = data['high']
        low = data['low']
        close = data['close']
        
        tr1 = high - low
        tr2 = abs(high - close.shift(1))
        tr3 = abs(low - close.shift(1))
        tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)

        if smoothing == 'WMA':
            atr = wma(tr, length)
        else:
            # Default to simple moving average if not WMA
            atr = tr.rolling(window=length).mean()
        return atr

    # Ensure no NaN values and reset index
    df.dropna(inplace=True)
    df.reset_index(drop=True, inplace=True)

    # Baseline calculation using HMA
    baseline_hma = hma(df['close'], baseline_length)

    # ATR calculation
    atr_values = calculate_atr(df, length=atr_period, smoothing=atr_smoothing)
    upper_band = df['close'] + atr_values * atr_mult
    lower_band = df['close'] - atr_values * atr_mult

    # SSL Channel calculations
    ema_high = hma(df['high'], baseline_length)
    ema_low = hma(df['low'], baseline_length)

    hlv = np.where(df['close'] > ema_high, 1, np.where(df['close'] < ema_low, -1, np.nan))
    hlv = pd.Series(hlv).ffill()
    
    ssl_down = np.where(hlv < 0, ema_high, ema_low)
    ssl_down_series = pd.Series(ssl_down, index=df.index)

    # Determine SSL Channel Color
    ssl_channel_color = np.where(df['close'] > ssl_down_series, 'blue', 
                                 np.where(df['close'] < ssl_down_series, 'red', np.nan))
    
    # Add SSL_Channel_Color to DataFrame
    df['SSL_Channel_Color'] = ssl_channel_color

    return df

def calculate_rma(df, length, column='close'):
    alpha = 1 / length
    rma = pd.Series(index=df.index, dtype=float)
    rma.iloc[length-1] = df[column].iloc[:length].mean()
    for i in range(length, len(df)):
        rma.iloc[i] = (df[column].iloc[i] * alpha) + (rma.iloc[i-1] * (1 - alpha))
    return rma

def calculate_atr(data, period=10):
    data['TR1'] = data['high'] - data['low']
    data['TR2'] = abs(data['high'] - data['close'].shift(1))
    data['TR3'] = abs(data['low'] - data['close'].shift(1))
    data['TR'] = data[['TR1', 'TR2', 'TR3']].max(axis=1)
    data['ATR'] = calculate_rma(data, length=period, column='TR')
    return data

def calculate_supertrend(df, period=10, multiplier=3.0):
    df = calculate_atr(df, period=period)
    hl2 = (df['high'] + df['low']) / 2
    df['basic_upperband'] = hl2 + (multiplier * df['ATR'])
    df['basic_lowerband'] = hl2 - (multiplier * df['ATR'])

    df['final_upperband'] = df['basic_upperband']
    df['final_lowerband'] = df['basic_lowerband']

    df['Trend'] = 1  # Start with an uptrend
    df['SuperTrend'] = np.nan

    for i in range(1, len(df)):
        curr_close = df['close'].iloc[i]
        prev_final_upperband = df['final_upperband'].iloc[i-1]
        prev_final_lowerband = df['final_lowerband'].iloc[i-1]
        prev_trend = df['Trend'].iloc[i-1]

        # Update final upper band
        if curr_close > prev_final_upperband:
            df['final_upperband'].iloc[i] = max(df['basic_upperband'].iloc[i], prev_final_upperband)
        else:
            df['final_upperband'].iloc[i] = df['basic_upperband'].iloc[i]

        # Update final lower band
        if curr_close < prev_final_lowerband:
            df['final_lowerband'].iloc[i] = min(df['basic_lowerband'].iloc[i], prev_final_lowerband)
        else:
            df['final_lowerband'].iloc[i] = df['basic_lowerband'].iloc[i]

        # Determine trend
        if prev_trend == 1:
            if curr_close < prev_final_lowerband:
                df['Trend'].iloc[i] = -1
            else:
                df['Trend'].iloc[i] = 1
        else:
            if curr_close > prev_final_upperband:
                df['Trend'].iloc[i] = 1
            else:
                df['Trend'].iloc[i] = -1

        # Set SuperTrend value
        if df['Trend'].iloc[i] == 1:
            df['SuperTrend'].iloc[i] = df['final_lowerband'].iloc[i]
        else:
            df['SuperTrend'].iloc[i] = df['final_upperband'].iloc[i]

    df['buy_signal'] = (df['Trend'] == 1) & (df['Trend'].shift(1) == -1)
    df['sell_signal'] = (df['Trend'] == -1) & (df['Trend'].shift(1) == 1)

    return df

def apply_supertrend(df, period=10, multiplier=3.0):
    df = calculate_supertrend(df, period, multiplier)
    df.drop(columns=['TR1', 'TR2', 'TR3', 'TR', 'basic_upperband', 'basic_lowerband'], inplace=True)
    return df

def volume_heatmap(df, length=60, slength=60,
                   threshold_extra_high=4.0,
                   threshold_high=2.5,
                   threshold_medium=1.0,
                   threshold_normal=-0.5):
    """
    Classify volume based on its deviation from the moving average
    """
    # Calculate moving average of volume
    mean = df['volume'].rolling(window=length).mean()

    # Calculate standard deviation of volume
    std = df['volume'].rolling(window=slength).std()

    # Calculate how many standard deviations volume is from mean
    stdbar = (df['volume'] - mean) / std
    df['stdbar'] = stdbar

    # Determine if price closed up or down
    direction = df['close'] > df['open']

    # Classify volume levels
    def classify_volume(row):
        if pd.isna(row['stdbar']):
            return 'Insufficient Data'

        if row['stdbar'] > threshold_extra_high:
            return 'Extra High Volume' + (' Up' if row['direction'] else ' Down')
        elif row['stdbar'] > threshold_high:
            return 'High Volume' + (' Up' if row['direction'] else ' Down')
        elif row['stdbar'] > threshold_medium:
            return 'Medium Volume' + (' Up' if row['direction'] else ' Down')
        elif row['stdbar'] > threshold_normal:
            return 'Normal Volume' + (' Up' if row['direction'] else ' Down')
        else:
            return 'Low Volume' + (' Up' if row['direction'] else ' Down')

    temp_df = pd.DataFrame({
        'stdbar': stdbar,
        'direction': direction
    })

    return temp_df.apply(classify_volume, axis=1)

import pandas as pd
import numpy as np

def large_cap_trend_meter2(df,
                          ema_fast=8,
                          ema_medium=21,
                          ema_slow=50,
                          ema_trend=200,
                          volume_threshold=1.2,
                          price_smoothing=2,
                          institutional_lookback=14,
                          sensitivity=1.2,
                          show_signals=True,
                          show_cloud_strength=True):
    """Enhanced EMA Trend Strength Meter for large-cap stocks"""
    
    result_df = df.copy()
    result_df = result_df.sort_index()

    # Calculate price source and smoothed price
    result_df['price_source'] = (result_df['high'] + result_df['low']) / 2
    result_df['smoothed_price'] = result_df['price_source'].rolling(window=price_smoothing).mean()

    # Calculate EMAs
    result_df['ema_fast_val'] = result_df['smoothed_price'].ewm(span=ema_fast, adjust=False).mean()
    result_df['ema_medium_val'] = result_df['smoothed_price'].ewm(span=ema_medium, adjust=False).mean()
    result_df['ema_slow_val'] = result_df['smoothed_price'].ewm(span=ema_slow, adjust=False).mean()
    result_df['ema_trend_val'] = result_df['smoothed_price'].ewm(span=ema_trend, adjust=False).mean()

    # Multi-layer EMA alignment
    result_df['fast_medium_bullish'] = result_df['ema_fast_val'] > result_df['ema_medium_val']
    result_df['medium_slow_bullish'] = result_df['ema_medium_val'] > result_df['ema_slow_val']
    result_df['slow_trend_bullish'] = result_df['ema_slow_val'] > result_df['ema_trend_val']

    result_df['alignment_count'] = result_df[['fast_medium_bullish', 'medium_slow_bullish', 'slow_trend_bullish']].sum(axis=1)
    result_df['perfect_alignment'] = result_df['alignment_count'] == 3

    # Distance-based strength calculation
    result_df['ema_separation_fast_medium'] = 100 * (result_df['ema_fast_val'] - result_df['ema_medium_val']) / result_df['close']
    result_df['ema_separation_medium_slow'] = 100 * (result_df['ema_medium_val'] - result_df['ema_slow_val']) / result_df['close']
    result_df['ema_separation_slow_trend'] = 100 * (result_df['ema_slow_val'] - result_df['ema_trend_val']) / result_df['close']

    max_separation_large_cap = 0.5
    result_df['total_separation'] = (result_df['ema_separation_fast_medium'].abs() +
                              result_df['ema_separation_medium_slow'].abs() +
                              result_df['ema_separation_slow_trend'].abs())
    result_df['separation_strength'] = np.minimum(40, (result_df['total_separation'] * 40 / (3 * max_separation_large_cap)) * sensitivity)

    result_df['alignment_strength'] = np.where(result_df['perfect_alignment'],
                                        result_df['separation_strength'],
                                        result_df['separation_strength'] * 0.6)

    # Volume-weighted momentum
    result_df['avg_volume'] = result_df['volume'].rolling(window=20).mean()
    result_df['volume_ratio'] = result_df['volume'] / result_df['avg_volume']
    result_df['volume_confirmed'] = result_df['volume_ratio'] >= volume_threshold

    result_df['ema_medium_momentum'] = 100 * (result_df['ema_medium_val'] - result_df['ema_medium_val'].shift(institutional_lookback)) / result_df['ema_medium_val'].shift(institutional_lookback)
    result_df['volume_weight'] = np.minimum(2.0, result_df['volume_ratio'])
    result_df['weighted_momentum'] = result_df['ema_medium_momentum'] * result_df['volume_weight']

    max_momentum_large_cap = 1.0
    result_df['momentum_strength'] = np.minimum(30, np.maximum(0, result_df['weighted_momentum'] * 30 / max_momentum_large_cap) * sensitivity)

    # EMA cloud strength
    result_df['cloud_1_strength'] = np.abs(result_df['ema_fast_val'] - result_df['ema_medium_val']) / result_df['close'] * 100
    result_df['cloud_2_strength'] = np.abs(result_df['ema_medium_val'] - result_df['ema_slow_val']) / result_df['close'] * 100
    result_df['cloud_3_strength'] = np.abs(result_df['ema_slow_val'] - result_df['ema_trend_val']) / result_df['close'] * 100

    result_df['cloud_1_bullish'] = result_df['ema_fast_val'] > result_df['ema_medium_val']
    result_df['cloud_2_bullish'] = result_df['ema_medium_val'] > result_df['ema_slow_val']
    result_df['cloud_3_bullish'] = result_df['ema_slow_val'] > result_df['ema_trend_val']

    result_df['cloud_consistency'] = np.where((result_df['cloud_1_bullish'] == result_df['cloud_2_bullish']) &
                                       (result_df['cloud_2_bullish'] == result_df['cloud_3_bullish']), 1.5, 1.0)

    result_df['avg_cloud_strength'] = (result_df['cloud_1_strength'] + result_df['cloud_2_strength'] + result_df['cloud_3_strength']) / 3
    result_df['normalized_cloud_strength'] = np.minimum(30, (result_df['avg_cloud_strength'] * 30 / 0.3) * result_df['cloud_consistency'] * sensitivity)

    # Combine all components
    result_df['total_trend_strength'] = result_df['alignment_strength'] + result_df['momentum_strength'] + result_df['normalized_cloud_strength']
    result_df['trend_strength'] = np.round(np.clip(result_df['total_trend_strength'], 0, 100))

    # Color coding for trend strength
    conditions_color = [
        result_df['trend_strength'] < 35,
        result_df['trend_strength'] < 65,
        result_df['trend_strength'] >= 65
    ]
    choices_color = [
        'crimson',
        'darkorange',
        'forestgreen'
    ]
    result_df['trend_color'] = np.select(conditions_color, choices_color, default='gray')

    # Create final dataframe
    final_df = df.copy()
    final_df['trend_strength'] = result_df['trend_strength']
    final_df['trend_color'] = result_df['trend_color']
    
    return final_df

def calculate_ema_200(df):
    """
    Calculate 200-period Exponential Moving Average from OHLCV data.
    
    Parameters:
    df (pd.DataFrame): DataFrame with OHLCV data. 
                      Expected columns: 'open', 'high', 'low', 'close', 'volume'
                      (case insensitive)
    
    Returns:
    pd.Series: 200-period EMA values indexed same as input DataFrame
    """
    
    # Make a copy to avoid modifying original DataFrame
    data = df.copy()
    
    # Normalize column names to lowercase for consistency
    data.columns = data.columns.str.lower()
    
    # Check if required 'close' column exists
    if 'close' not in data.columns:
        raise ValueError("DataFrame must contain 'close' column for EMA calculation")
    
    # Calculate 200-period EMA using pandas ewm function
    # Alpha = 2 / (period + 1) = 2 / (200 + 1) ≈ 0.00995
    ema_200 = data['close'].ewm(span=200, adjust=False).mean()
    
    return ema_200

def WMA(series, period):
    weights = np.arange(1, period + 1)
    return series.rolling(period).apply(lambda x: np.dot(x, weights) / weights.sum(), raw=True)

def compute_ema_trend(df, emaLength=50, minThreshold=0.5, maxThreshold=5.0, smoothPeriods=3):
    """
    Computes the Smoothed EMA Strength Oscillator and adds 'ema_trend' to the DataFrame.

    Parameters:
    - df: pandas.DataFrame with columns including 'close'
    - emaLength: int, default 50
    - minThreshold: float, default 0.5 (%)
    - maxThreshold: float, default 5.0 (%)
    - smoothPeriods: int, default 3

    Returns:
    - df: pandas.DataFrame with added 'ema_trend' column
    """
    df = df.copy()  # Avoid modifying the original DataFrame

    # Calculate EMA of the close prices
    df['ema'] = df['close'].ewm(span=emaLength, adjust=False).mean()

    # Calculate raw distance (% difference between close and EMA)
    df['rawDistance'] = (df['close'] - df['ema']) / df['ema'] * 100

    # Smooth the raw distance using WMA
    df['distance'] = WMA(df['rawDistance'], smoothPeriods)

    # Initialize rawOsc with zeros
    df['rawOsc'] = 0.0

    # Apply conditions to compute rawOsc
    mask = df['distance'] >= minThreshold
    df.loc[mask, 'normalized'] = (df.loc[mask, 'distance'] - minThreshold) / (maxThreshold - minThreshold)
    df.loc[mask, 'normalized'] = df.loc[mask, 'normalized'].clip(upper=1)  # Ensure normalized <= 1
    df.loc[mask, 'rawOsc'] = 8 + df.loc[mask, 'normalized'] * 2

    # Smooth the rawOsc to get the final oscillator value
    df['ema_trend'] = WMA(df['rawOsc'], smoothPeriods)

    # Clean up intermediate columns if desired
    df.drop(['ema', 'rawDistance', 'distance', 'rawOsc', 'normalized'], axis=1, inplace=True, errors='ignore')

    return df

# TRADING CODE

In [ ]:
import asyncio
from ib_insync import *
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
from pymongo import MongoClient
from bson import ObjectId
import warnings

# Method 1: Suppress all warnings
warnings.filterwarnings("ignore")

# Method 2: Suppress specific warning types
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Connect to MongoDB
client = MongoClient('mongodb://localhost:27017/')
db = client['trading_db']
trades_collection = db['trades_v2']
exclude_tickers_collection = db['excluded_tickers_v2']  # New collection to track tickers

# Connect to IB
util.startLoop()
ib = IB()
ib.connect('127.0.0.1', 7497, clientId=random.randint(1, 1000))  # Update port and clientId if necessary

tickers =  ['AAPL', 'TSLA', 'AMZN', 'GOOG', 'COIN', 'NVDA', 'BA','TSM'] #['TSLA', 'COIN' ] #
contracts = [Stock(ticker, 'SMART', 'USD') for ticker in tickers]

# Ensure contracts are resolved
ib.qualifyContracts(*contracts)

# Update the create_trade_document function to include the new fields
def create_trade_document(symbol, direction, entry_price, quantity):
    return {
        'instrument': symbol,
        'direction': direction,
        'entry_price': entry_price,
        'highest_price': entry_price,  # Initialize the highest price with the entry price
        'quantity': quantity,
        'current_quantity': quantity,  # New field to track remaining quantity
        'partial_exits': [],           # New field to track partial exit information
        'entry_timestamp': datetime.now(),
        'order_id': str(ObjectId()),
        'exit_price': None,
        'exit_timestamp': None,
        'reason': None,
        'current_pnl': 0,
        'realized_pnl': 0,
        'status': 'open',
        'created_at': datetime.now(),
        'updated_at': datetime.now()
    }

# Helper function to update trade document
def update_trade_document(trade_id, updates):
    updates['updated_at'] = datetime.now()
    trades_collection.update_one(
        {'_id': ObjectId(trade_id)},
        {'$set': updates}
    )

async def check_and_trade():
    # Configuration parameters for position management
    base_trailing_amount = 1.0  # Base trailing stop amount in dollars
    min_trailing_pct = 0.005  # Minimum trailing stop as percentage of price (0.5%)
    atr_multiplier = 1.5  # ATR multiplier for dynamic trailing stops
    
    # Stop loss parameters
    stop_loss_1 = 0.01  # 1% stop loss
    stop_loss_2 = 0.02  # 2% stop loss
    
    # Partial profit-taking levels
    profit_level_1 = 0.01  # 1% profit
    profit_level_2 = 0.02  # 2% profit
    profit_level_3 = 0.03  # 3% profit
    
    # Partial position sizing for exits
    partial_size_1 = 0.25  # Exit 25% at first profit level
    partial_size_2 = 0.25  # Exit 25% at second profit level
    partial_size_3 = 0.25  # Exit 25% at third profit level
    # Remaining 25% follows trailing stop
    
    while True:
        # Fetch current positions
        positions = ib.positions()
        position_sizes = {pos.contract.symbol: pos.position for pos in positions}

        # Fetch open orders
        open_orders = ib.reqOpenOrders()
        order_symbols = {order.contract.symbol for order in open_orders}

        for contract in contracts:
            symbol = contract.symbol
            print(f"Evaluating contract for {symbol}")

            # Fetch historical data
            bars = ib.reqHistoricalData(
                contract,
                endDateTime='',
                durationStr='2 D',
                barSizeSetting='5 mins',
                whatToShow='TRADES',
                useRTH=False,
                formatDate=1
            )

            if not bars:
                print(f"No historical data for {symbol}")
                continue

            # Convert to DataFrame
            df = util.df(bars)

            # Apply technical indicators
            df = add_ssl_channel_color(df)
            df['Volume_Label'] = volume_heatmap(df)
            df = calculate_qqe_signal(df)
            df = apply_supertrend(df)
            df = large_cap_trend_meter2(df)
            df['ema_200'] = calculate_ema_200(df)

            
            # Calculate ATR for dynamic trailing stop
            if 'ATR' not in df.columns:
                df = calculate_atr(df, period=14)  # Use 14-period ATR

            current_row = df.iloc[-1]
            current_price = current_row['close']
            current_atr = current_row.get('ATR', base_trailing_amount)  # Fallback if ATR calculation fails
            current_trend_color = current_row.get('trend_color', 'unknown')
            
            # Check for open trades
            open_trade = trades_collection.find_one({
                'instrument': symbol,
                'status': 'open'
            })

            # Get current date
            current_date = datetime.now().date()

            # Position management
            if open_trade:
                entry_price = open_trade['entry_price']
                highest_price = open_trade.get('highest_price', entry_price)
                original_quantity = open_trade['quantity']
                current_quantity = open_trade.get('current_quantity', original_quantity)
                partial_exits = open_trade.get('partial_exits', [])
                time_in_trade = (datetime.now() - open_trade['entry_timestamp']).total_seconds() / 3600  # Hours
                
                # Update highest price if current price is higher
                if current_price > highest_price:
                    highest_price = current_price
                    update_trade_document(open_trade['_id'], {
                        'highest_price': highest_price
                    })

                # Calculate profit/loss as a percentage
                profit_pct = (current_price - entry_price) / entry_price
                profit_loss_per_share = current_price - entry_price
                current_pnl = profit_loss_per_share * current_quantity
                
                # Update current PnL
                update_trade_document(open_trade['_id'], {
                    'current_pnl': current_pnl
                })
                
                # NEW STOP LOSS LOGIC BASED ON TREND COLOR
                should_exit_on_loss = False
                exit_reason = None
                
                if profit_pct <= -stop_loss_1:  # At 1% loss
                    if current_trend_color != 'forestgreen':
                        # Trend color is not forestgreen, exit immediately
                        should_exit_on_loss = True
                        exit_reason = 'stop_loss_1pct_trend_change'
                        print(f"Exiting {symbol} at 1% loss due to trend color change: {current_trend_color}")
                    elif profit_pct <= -stop_loss_2:  # At 2% loss
                        # Even if trend is still forestgreen, exit at 2% loss
                        should_exit_on_loss = True
                        exit_reason = 'stop_loss_2pct_max'
                        print(f"Exiting {symbol} at 2% loss (maximum allowed)")
                    else:
                        print(f"Holding {symbol} at {profit_pct:.2%} loss - trend still forestgreen")
                
                # Execute stop loss exit if conditions are met
                if should_exit_on_loss and current_quantity > 0:
                    order = MarketOrder('SELL', current_quantity)
                    trade = ib.placeOrder(contract, order)

                    # Calculate average exit price considering partial exits
                    total_value = current_price * current_quantity
                    total_quantity = current_quantity
                    
                    for exit in partial_exits:
                        total_value += exit['price'] * exit['quantity']
                        total_quantity += exit['quantity']
                    
                    avg_exit_price = total_value / total_quantity if total_quantity > 0 else current_price
                    
                    # Calculate final P&L
                    realized_pnl = (avg_exit_price - entry_price) * original_quantity

                    # Update trade document
                    update_trade_document(open_trade['_id'], {
                        'exit_price': avg_exit_price,
                        'exit_timestamp': datetime.now(),
                        'reason': exit_reason,
                        'realized_pnl': realized_pnl,
                        'status': 'closed',
                        'current_quantity': 0
                    })

                    print(f"Stop loss executed for {symbol} at {current_price}, avg exit: {avg_exit_price}")
                    continue  # Skip to next symbol since we've exited
                
                # Only proceed with profit-taking and trailing stops if we're not in a loss situation
                # or if we're in a loss but trend is still good (forestgreen) and loss is less than 2%
                
                # Determine dynamic trailing stop (only if not in loss territory)
                if profit_pct > 0:
                    # Use max of different methods to create adaptive trailing stop
                    atr_based_stop = highest_price - (current_atr * atr_multiplier)
                    percentage_based_stop = highest_price * (1 - min_trailing_pct)
                    fixed_based_stop = highest_price - base_trailing_amount
                    
                    # Tighten trail the longer we're in the trade
                    time_factor = min(1.0, time_in_trade / 24)  # Full tightening after 24 hours
                    trailing_stop_price = atr_based_stop * (1 - 0.25 * time_factor)  # Tighten by up to 25%
                    
                    # Use the tightest (highest) of the calculated stops
                    trailing_stop_price = max(trailing_stop_price, percentage_based_stop, fixed_based_stop)
                    
                    # Adjust trailing stop based on trend strength
                    if current_row['Trend'] == 1 and current_row['QQE_Signal'] == 'Up':
                        # Strong uptrend, give more room
                        trailing_stop_price = trailing_stop_price * 0.98  # 2% more room
                    
                    # Take partial profits at different levels
                    if profit_pct >= profit_level_3 and not any(exit['level'] == 3 for exit in partial_exits):
                        # Take third partial profit
                        exit_quantity = int(original_quantity * partial_size_3)
                        if exit_quantity > 0 and current_quantity >= exit_quantity:
                            order = MarketOrder('SELL', exit_quantity)
                            trade = ib.placeOrder(contract, order)
                            
                            partial_exits.append({
                                'level': 3,
                                'price': current_price,
                                'quantity': exit_quantity,
                                'timestamp': datetime.now()
                            })
                            
                            current_quantity -= exit_quantity
                            update_trade_document(open_trade['_id'], {
                                'current_quantity': current_quantity,
                                'partial_exits': partial_exits
                            })
                            
                            print(f"Third partial profit taken for {symbol} at {current_price} ({profit_pct:.2%})")
                    
                    elif profit_pct >= profit_level_2 and not any(exit['level'] == 2 for exit in partial_exits):
                        # Take second partial profit
                        exit_quantity = int(original_quantity * partial_size_2)
                        if exit_quantity > 0 and current_quantity >= exit_quantity:
                            order = MarketOrder('SELL', exit_quantity)
                            trade = ib.placeOrder(contract, order)
                            
                            partial_exits.append({
                                'level': 2,
                                'price': current_price,
                                'quantity': exit_quantity,
                                'timestamp': datetime.now()
                            })
                            
                            current_quantity -= exit_quantity
                            update_trade_document(open_trade['_id'], {
                                'current_quantity': current_quantity,
                                'partial_exits': partial_exits
                            })
                            
                            print(f"Second partial profit taken for {symbol} at {current_price} ({profit_pct:.2%})")
                    
                    elif profit_pct >= profit_level_1 and not any(exit['level'] == 1 for exit in partial_exits):
                        # Take first partial profit
                        exit_quantity = int(original_quantity * partial_size_1)
                        if exit_quantity > 0 and current_quantity >= exit_quantity:
                            order = MarketOrder('SELL', exit_quantity)
                            trade = ib.placeOrder(contract, order)
                            
                            partial_exits.append({
                                'level': 1,
                                'price': current_price,
                                'quantity': exit_quantity,
                                'timestamp': datetime.now()
                            })
                            
                            current_quantity -= exit_quantity
                            update_trade_document(open_trade['_id'], {
                                'current_quantity': current_quantity,
                                'partial_exits': partial_exits
                            })
                            
                            print(f"First partial profit taken for {symbol} at {current_price} ({profit_pct:.2%})")

                    # Check for trailing stop exit with remaining position (only when in profit)
                    if current_price <= trailing_stop_price and current_quantity > 0:
                        order = MarketOrder('SELL', current_quantity)
                        trade = ib.placeOrder(contract, order)

                        # Calculate average exit price considering partial exits
                        total_value = current_price * current_quantity
                        total_quantity = current_quantity
                        
                        for exit in partial_exits:
                            total_value += exit['price'] * exit['quantity']
                            total_quantity += exit['quantity']
                        
                        avg_exit_price = total_value / total_quantity if total_quantity > 0 else current_price
                        
                        # Calculate final P&L
                        realized_pnl = (avg_exit_price - entry_price) * original_quantity

                        # Update trade document
                        update_trade_document(open_trade['_id'], {
                            'exit_price': avg_exit_price,
                            'exit_timestamp': datetime.now(),
                            'reason': 'trailing_stop_hit',
                            'realized_pnl': realized_pnl,
                            'status': 'closed',
                            'current_quantity': 0
                        })

                        print(f"Trailing stop loss executed for {symbol} at {current_price}, avg exit: {avg_exit_price}")

            # Entry management (same as before)
            else:
                # Check if we have already traded this ticker today
                exclude_entry = exclude_tickers_collection.find_one({
                    'ticker': symbol,
                    'date': current_date.isoformat()
                })

                if exclude_entry:
                    print(f"Already traded {symbol} today. Skipping new trade.")
                    continue  # Skip to next ticker

                # Check for buy signal
                if (current_row['SSL_Channel_Color'] == 'blue' and
                    #current_row['Trend'] == 1 and
                    #current_row['QQE_Signal'] == 'Up'
                    current_row['close'] > current_row['ema_200'] and
                    current_row['trend_color'] == 'forestgreen' ):

                    order_size = 20  # Adjust as needed
                    order = MarketOrder('BUY', order_size)
                    trade = ib.placeOrder(contract, order)

                    # Create new trade document with additional fields
                    trade_doc = create_trade_document(
                        symbol=symbol,
                        direction='long',
                        entry_price=current_price,
                        quantity=order_size
                    )
                    
                    # Add new fields for the improved trailing stop system
                    trade_doc['current_quantity'] = order_size
                    trade_doc['partial_exits'] = []
                    
                    trades_collection.insert_one(trade_doc)

                    # Add ticker to exclude_tickers collection
                    exclude_tickers_collection.insert_one({
                        'ticker': symbol,
                        'date': current_date.isoformat()
                    })

                    print(f"New long position opened for {symbol} at {current_price}")

            # Sleep briefly to avoid pacing violations
            await asyncio.sleep(1)

        # Wait for 5 minutes before next iteration
        print("Waiting for 5 minutes...")
        await asyncio.sleep(300)

# Run the async loop
try:
    ib.run(check_and_trade())
except Exception as e:
    print(f"An error occurred: {e}")
finally:
    ib.disconnect()
    print("Disconnected from IB.")

In [ ]:
import asyncio
from ib_insync import *
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
from pymongo import MongoClient
from bson import ObjectId
import warnings

# Method 1: Suppress all warnings
warnings.filterwarnings("ignore")

# Method 2: Suppress specific warning types
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Connect to MongoDB
client = MongoClient('mongodb://localhost:27017/')
db = client['trading_db']
trades_collection = db['trades_v2']
exclude_tickers_collection = db['excluded_tickers_v2']  # New collection to track tickers

# Connect to IB
util.startLoop()
ib = IB()
ib.connect('127.0.0.1', 7497, clientId=random.randint(1, 1000))  # Update port and clientId if necessary

tickers =  ['AAPL', 'TSLA', 'AMZN', 'GOOG', 'COIN', 'NVDA', 'BA','TSM'] #['TSLA', 'COIN' ] #
contracts = [Stock(ticker, 'SMART', 'USD') for ticker in tickers]

# Ensure contracts are resolved
ib.qualifyContracts(*contracts)

# Update the create_trade_document function - simplified without partial exit fields
def create_trade_document(symbol, direction, entry_price, quantity):
    return {
        'instrument': symbol,
        'direction': direction,
        'entry_price': entry_price,
        'highest_price': entry_price,  # Initialize the highest price with the entry price
        'quantity': quantity,
        'entry_timestamp': datetime.now(),
        'order_id': str(ObjectId()),
        'exit_price': None,
        'exit_timestamp': None,
        'reason': None,
        'current_pnl': 0,
        'realized_pnl': 0,
        'status': 'open',
        'created_at': datetime.now(),
        'updated_at': datetime.now()
    }

# Helper function to update trade document
def update_trade_document(trade_id, updates):
    updates['updated_at'] = datetime.now()
    trades_collection.update_one(
        {'_id': ObjectId(trade_id)},
        {'$set': updates}
    )

async def check_and_trade():
    # Configuration parameters for position management
    base_trailing_amount = 1.0  # Base trailing stop amount in dollars
    min_trailing_pct = 0.005  # Minimum trailing stop as percentage of price (0.5%)
    atr_multiplier = 1.5  # ATR multiplier for dynamic trailing stops
    
    # Stop loss parameters
    stop_loss_1 = 0.01  # 1% stop loss
    stop_loss_2 = 0.02  # 2% stop loss
    
    # Take profit parameters - simplified
    take_profit_pct = 0.01  # 1% profit
    take_profit_dollar = 2.0  # $2 profit per share
    
    while True:
        # Fetch current positions
        positions = ib.positions()
        position_sizes = {pos.contract.symbol: pos.position for pos in positions}

        # Fetch open orders
        open_orders = ib.reqOpenOrders()
        order_symbols = {order.contract.symbol for order in open_orders}

        for contract in contracts:
            symbol = contract.symbol
            print(f"Evaluating contract for {symbol}")

            # Fetch historical data
            bars = ib.reqHistoricalData(
                contract,
                endDateTime='',
                durationStr='2 D',
                barSizeSetting='5 mins',
                whatToShow='TRADES',
                useRTH=False,
                formatDate=1
            )

            if not bars:
                print(f"No historical data for {symbol}")
                continue

            # Convert to DataFrame
            df = util.df(bars)

            # Apply technical indicators
            df = add_ssl_channel_color(df)
            df['Volume_Label'] = volume_heatmap(df)
            df = calculate_qqe_signal(df)
            df = apply_supertrend(df)
            df = large_cap_trend_meter2(df)
            df['ema_200'] = calculate_ema_200(df)
            df = compute_ema_trend(df)

            print(df.head())

            # Calculate ATR for dynamic trailing stop
            if 'ATR' not in df.columns:
                df = calculate_atr(df, period=14)  # Use 14-period ATR

            current_row = df.iloc[-1]
            current_price = current_row['close']
            current_atr = current_row.get('ATR', base_trailing_amount)  # Fallback if ATR calculation fails
            current_trend_color = current_row.get('trend_color', 'unknown')
            
            # Check for open trades
            open_trade = trades_collection.find_one({
                'instrument': symbol,
                'status': 'open'
            })

            # Get current date
            current_date = datetime.now().date()

            # Position management
            if open_trade:
                entry_price = open_trade['entry_price']
                highest_price = open_trade.get('highest_price', entry_price)
                quantity = open_trade['quantity']
                time_in_trade = (datetime.now() - open_trade['entry_timestamp']).total_seconds() / 3600  # Hours
                
                # Update highest price if current price is higher
                if current_price > highest_price:
                    highest_price = current_price
                    update_trade_document(open_trade['_id'], {
                        'highest_price': highest_price
                    })

                # Calculate profit/loss as a percentage and dollar amount
                profit_pct = (current_price - entry_price) / entry_price
                profit_loss_per_share = current_price - entry_price
                current_pnl = profit_loss_per_share * quantity
                
                # Update current PnL
                update_trade_document(open_trade['_id'], {
                    'current_pnl': current_pnl
                })
                
                # STOP LOSS LOGIC BASED ON TREND COLOR
                should_exit_on_loss = False
                exit_reason = None
                
                if profit_pct <= -stop_loss_1:  # At 1% loss
                    if current_trend_color != 'forestgreen':
                        # Trend color is not forestgreen, exit immediately
                        should_exit_on_loss = True
                        exit_reason = 'stop_loss_1pct_trend_change'
                        print(f"Exiting {symbol} at 1% loss due to trend color change: {current_trend_color}")
                    elif profit_pct <= -stop_loss_2:  # At 2% loss
                        # Even if trend is still forestgreen, exit at 2% loss
                        should_exit_on_loss = True
                        exit_reason = 'stop_loss_2pct_max'
                        print(f"Exiting {symbol} at 2% loss (maximum allowed)")
                    else:
                        print(f"Holding {symbol} at {profit_pct:.2%} loss - trend still forestgreen")
                
                # Execute stop loss exit if conditions are met
                if should_exit_on_loss and quantity > 0:
                    order = MarketOrder('SELL', quantity)
                    trade = ib.placeOrder(contract, order)

                    # Calculate final P&L
                    realized_pnl = (current_price - entry_price) * quantity

                    # Update trade document
                    update_trade_document(open_trade['_id'], {
                        'exit_price': current_price,
                        'exit_timestamp': datetime.now(),
                        'reason': exit_reason,
                        'realized_pnl': realized_pnl,
                        'status': 'closed'
                    })

                    print(f"Stop loss executed for {symbol} at {current_price}")
                    continue  # Skip to next symbol since we've exited
                
                # TAKE PROFIT LOGIC - Exit 100% of position when profit is 1% OR $2 per share
                if profit_pct >= take_profit_pct or profit_loss_per_share >= take_profit_dollar:
                    order = MarketOrder('SELL', quantity)
                    trade = ib.placeOrder(contract, order)

                    # Calculate final P&L
                    realized_pnl = (current_price - entry_price) * quantity

                    # Update trade document
                    update_trade_document(open_trade['_id'], {
                        'exit_price': current_price,
                        'exit_timestamp': datetime.now(),
                        'reason': f'take_profit_{profit_pct:.2%}_or_${profit_loss_per_share:.2f}',
                        'realized_pnl': realized_pnl,
                        'status': 'closed'
                    })

                    print(f"Take profit executed for {symbol} at {current_price} (Profit: {profit_pct:.2%} or ${profit_loss_per_share:.2f}/share)")
                    continue  # Skip to next symbol since we've exited
                
                # TRAILING STOP LOGIC - Only applies if we haven't hit take profit yet
                # Determine dynamic trailing stop (only if in profit territory)
                if profit_pct > 0:
                    # Use max of different methods to create adaptive trailing stop
                    atr_based_stop = highest_price - (current_atr * atr_multiplier)
                    percentage_based_stop = highest_price * (1 - min_trailing_pct)
                    fixed_based_stop = highest_price - base_trailing_amount
                    
                    # Tighten trail the longer we're in the trade
                    time_factor = min(1.0, time_in_trade / 24)  # Full tightening after 24 hours
                    trailing_stop_price = atr_based_stop * (1 - 0.25 * time_factor)  # Tighten by up to 25%
                    
                    # Use the tightest (highest) of the calculated stops
                    trailing_stop_price = max(trailing_stop_price, percentage_based_stop, fixed_based_stop)
                    
                    # Adjust trailing stop based on trend strength
                    if current_row['Trend'] == 1 and current_row['QQE_Signal'] == 'Up':
                        # Strong uptrend, give more room
                        trailing_stop_price = trailing_stop_price * 0.98  # 2% more room

                    # Check for trailing stop exit
                    if current_price <= trailing_stop_price and quantity > 0:
                        order = MarketOrder('SELL', quantity)
                        trade = ib.placeOrder(contract, order)

                        # Calculate final P&L
                        realized_pnl = (current_price - entry_price) * quantity

                        # Update trade document
                        update_trade_document(open_trade['_id'], {
                            'exit_price': current_price,
                            'exit_timestamp': datetime.now(),
                            'reason': 'trailing_stop_hit',
                            'realized_pnl': realized_pnl,
                            'status': 'closed'
                        })

                        print(f"Trailing stop executed for {symbol} at {current_price}")

            # Entry management (same as before)
            else:
                # Check if we have already traded this ticker today
                exclude_entry = exclude_tickers_collection.find_one({
                    'ticker': symbol,
                    'date': current_date.isoformat()
                })

                if exclude_entry:
                    print(f"Already traded {symbol} today. Skipping new trade.")
                    continue  # Skip to next ticker

                # Check for buy signal
                #if (current_row['SSL_Channel_Color'] == 'blue' and
                #    #current_row['Trend'] == 1 and
                #    #current_row['QQE_Signal'] == 'Up'
                #    current_row['close'] > current_row['ema_200'] and
                #    current_row['trend_color'] == 'forestgreen' ):

                if (current_row['SSL_Channel_Color'] == 'blue' and
                    #current_row['Trend'] == 1 and
                    #current_row['QQE_Signal'] == 'Up'
                    current_row['close'] > current_row['ema_200'] and
                    current_row['ema_trend'] >= 8.0 ):

                    order_size = 20  # Adjust as needed
                    order = MarketOrder('BUY', order_size)
                    trade = ib.placeOrder(contract, order)

                    # Create new trade document - simplified
                    trade_doc = create_trade_document(
                        symbol=symbol,
                        direction='long',
                        entry_price=current_price,
                        quantity=order_size
                    )
                    
                    trades_collection.insert_one(trade_doc)

                    # Add ticker to exclude_tickers collection
                    exclude_tickers_collection.insert_one({
                        'ticker': symbol,
                        'date': current_date.isoformat()
                    })

                    print(f"New long position opened for {symbol} at {current_price}")

            # Sleep briefly to avoid pacing violations
            await asyncio.sleep(1)

        # Wait for 5 minutes before next iteration
        print("Waiting for 5 minutes...")
        await asyncio.sleep(300)

# Run the async loop
try:
    ib.run(check_and_trade())
except Exception as e:
    print(f"An error occurred: {e}")
finally:
    ib.disconnect()
    print("Disconnected from IB.")